# PINN — Tamil Nadu Groundwater Level

Trains a physics-informed neural network for the 2D transient unconfined groundwater flow equation:

`S(x,y) * dh/dt = div(T(x,y) * grad(h)) + R(x,y,t) - Q(x,y,t)`

`S`, `T` are learned per-district scalars (inverse problem). Inputs are the Phase 1 outputs
`observations.csv` and `collocation.csv`, read from the Kaggle dataset
`benadictinfanta/water-data` (see the config cell for `DATA_DIR`) — see `DATA_DICTIONARY.md` and
`STATUS.md` in the project repo for what each column means and every known data-quality caveat
(sign convention, missingness, etc).

**Run this on Kaggle with a GPU and internet access enabled** — it was authored on a CPU-only
machine and not run there. `DATA_DIR`/`OUTPUT_DIR` are set to Kaggle paths permanently; for Colab,
change them per the comment in the config cell.


In [ ]:
# --- GPU/torch compatibility guard (Run 21) ---------------------------------
# Kaggle's bundled torch (2.10.0+cu128) dropped support for cc < 7.0 GPUs. The
# free GPU pool sometimes hands out a Tesla P100 (sm_60, cc 6.0), which then
# dies with "CUDA error: no kernel image is available" on the first matmul.
# Detect the device's compute capability in a throwaway subprocess (so we don't
# import torch into the main session prematurely) and, only if it is < 7.0,
# install a torch build that still ships sm_60 kernels. T4/A100/L4 (cc >= 7.0)
# are untouched and keep the current torch with an identical run.
import subprocess, sys
det = subprocess.run(
    [sys.executable, "-c",
     "import torch as t; "
     "print('CAP', t.cuda.get_device_capability(0) if t.cuda.is_available() else 'cpu')"],
    capture_output=True, text=True, timeout=120)
print("torch probe:", det.stdout.strip() or det.stderr.strip().splitlines()[-1])
cap = None
if det.returncode == 0:
    for tok in (det.stdout or "").split():
        if tok.startswith("CAP"):
            continue
        try:
            cap = tuple(int(x) for x in tok.strip("() ").split(",") if x.strip())
            break
        except ValueError:
            pass
if cap is None:
    print("no usable GPU/capability -- leaving bundled torch in place")
elif cap[0] < 7:
    print(f"GPU cc {cap} needs sm_60 support: installing torch 2.5.1+cu118 "
          f"(supports cc 5.0-9.0) BEFORE importing torch in this session")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "torch==2.5.1",
                        "--index-url", "https://download.pytorch.org/whl/cu118"],
                       text=True, timeout=1800)
    verify = subprocess.run([sys.executable, "-c",
                             "import torch; print('now', torch.__version__)"],
                            capture_output=True, text=True, timeout=300)
    print("verify:", (verify.stdout or verify.stderr).strip())
else:
    print(f"GPU cc {cap} >= 7: bundled torch stays (unchanged behavior for cc>=70 runs)")


In [ ]:
import os
import math
import time
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__}, device = {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Config

Single path variable + all hyperparameters/weights up top, nothing buried later.

In [ ]:
# --- paths --------------------------------------------------------------
# Kaggle paths, set permanently (dataset: benadictinfanta/water-data).
# For Colab instead: DATA_DIR = "/content/drive/MyDrive/pinn_predict/data/processed"
DATA_DIR = "/kaggle/input/datasets/benadictinfanta/water-data"
OBSERVATIONS_PATH = os.path.join(DATA_DIR, "observations.csv")
COLLOCATION_PATH = os.path.join(DATA_DIR, "collocation.csv")
OUTPUT_DIR = "/kaggle/working/"

# --- elevation (depth -> head conversion, see STATUS.md issue 2/4) --------
# No DEM ships in data/raw/, so this is fetched at runtime from a free SRTM
# API rather than added to the Phase 1 pipeline -- see the "Elevation lookup"
# cell for why. Cached to ELEVATION_CACHE_PATH so re-runs (e.g. after
# restarting the kernel) don't re-fetch ~4,000 points from scratch.
ELEVATION_API_URL = "https://api.opentopodata.org/v1/srtm30m"
ELEVATION_BATCH_SIZE = 100      # public server limit
ELEVATION_RATE_LIMIT_S = 1.0    # public server limit: 1 request/second
ELEVATION_MAX_RETRIES = 3
ELEVATION_CACHE_PATH = os.path.join(OUTPUT_DIR, "elevation_cache.csv")

# --- model architecture ---------------------------------------------------
EMBEDDING_DIM = 8
# Run 19: per-well baseline as an INPUT FEATURE instead of a learned embedding.
# Run 18's per-well embedding (WELL_EMBEDDING_DIM) was REVERTED after it overfit: the net
# memorized each well's identity on the train window and extrapolated to 383m beyond the
# 100m H_MAX clip (see STATUS.md eighteenth-run entry).  The thing persistence (5.50m, the
# forecast baseline) actually knows is each well's LAST OBSERVED DEPTH BEFORE THE CUTOFF --
# so we hand the network that number directly (h0, normalized as h0hat) instead of making
# it re-learn per-well baselines as free parameters.  The net then only has to model the
# trend/seasonality ON TOP of the anchor, which is far easier than reconstructing baselines
# from (x,y,elev).  Held-out wells and collocation points have NO anchor (0 in normalized
# units = the global mean), so the spatial task is unchanged and stays honest.
H0_DROPOUT_P = 0.3        # Run 20: fraction of TRAINING rows whose anchor is zeroed (both the
                          # input slot AND the additive residual).  Run 19 leaned hard on the
                          # anchor (first-layer weight 0.334 vs ~0.20 on every other input), so
                          # held-out wells (h0hat=0) were out-of-distribution and the spatial
                          # metric regressed to 13.45m.  Dropout makes the unanchored case a
                          # regular part of training -- see the model cell for the residual form.
HIDDEN_WIDTH = 64               # cut from 128 after the "accuracy" run (RMSE 12.49m, just barely
                                 # losing to the 12.07m district-mean baseline): pred-vs-actual showed
                                 # smooth looping bands instead of a noisy cloud around the diagonal --
                                 # a sign the 101K-param net is riding a low-dimensional shortcut
                                 # (embedding + a smooth function of t) rather than resolving real
                                 # well-to-well variation, which (x,y) alone can't fully explain anyway
                                 # (STATUS.md issue 4 -- h is depth, not head, so local ground elevation
                                 # leaks into "spatial" variation no smooth field can capture). Smaller
                                 # net + more weight decay, re-test RMSE against the same baseline.
                                 # The fifth run (2026-07-19) showed this alone doesn't close the gap --
                                 # see the elevation/head cells below, which address issue 4 directly.
NUM_HIDDEN_LAYERS = 4            # cut from 7, same reasoning

# --- training h range (a MODELING choice, logged loudly -- see STATUS.md) --
# Phase 1's clean_groundwater.py drops |h| >= 200 as sentinels, which is too loose:
# h is p50=6.1m, p99=54m, but the surviving range runs to +-199.97m. Those 0.08% of
# rows contribute ~29 to the MSE on their own (~5.4m of RMSE) and are not physically
# plausible depth-to-water for Tamil Nadu. Rows outside this range are EXCLUDED from
# training and evaluation, and the count is printed. Widen/narrow and re-run to see
# the sensitivity; this is not a claim that every excluded row is bad data.
# NOTE: this filter is applied to raw depth `h`, before the elevation/head conversion
# below -- the sentinel cut is about bad readings, not about head vs. depth.
H_MIN = -5.0                   # small negative = artesian/near-surface, plausible (median
                                # negative is only -1.9m); the -65m..-199m tail is not.
H_MAX = 100.0

# --- loss weights (kept explicit, not buried in the loss function) --------
W_DATA = 1.0
W_PDE = 0.0001                   # PDE loss weight
                                # Run 11 raised it 373x alongside the head->depth target switch, to
                                # "hold the physics/data balance constant". That reasoning was wrong.
                                # It preserved the ratio of loss VALUES, but in Run 10 the PDE was
                                # already ~98% of the total loss and merely INERT (its curve was flat
                                # noise -- an ineffective gradient). Preserving that dominance was
                                # only safe while it stayed inert. The target switch made the PDE
                                # gradient effective (Run 11's pde loss actually fell, 1000 -> 70),
                                # and the preserved dominance then overrode the data fit: data loss
                                # dipped to ~0.32 by step ~1000, then ROSE to ~0.55 and plateaued.
                                # At 1e-4 with the depth target the PDE contributes ~10% of the total
                                # instead of ~82% -- sub-dominant, which is what it should have been.
                                # See STATUS.md eleventh-run entry.
                                # tuning history (held-out wells, same split every run):
                                # Pre-head-conversion runs (depth target -- PDE was wrong quantity):
                                #   1e-4 -> RMSE 15.6m, T frozen at softplus(0) for all 38 districts
                                #   1e-2 -> RMSE 18.3m, T spread (0.52-1.65) for 26 districts with R
                                #   1e-3 -> RMSE 18.1m, T mostly back at init -- worst of both
                                #   ALL THREE lose to the district-mean baseline (RMSE 12.07m).
                                # Post-head-conversion runs (head = elev-h target, PDE now correct):
                                #   1e-6 -> RMSE 11.78m, MAE 6.77m -- first run to beat baseline
                                #           convincingly; S/T frozen (W_PDE negligible), Run 7.
                                #   1e-4 -> RMSE 11.86m, MAE 7.02m (Run 8, 2026-07-24): still beats
                                #           baseline but T stayed BYTE-IDENTICAL to Run 7's frozen
                                #           softplus(0) despite the 100x weight increase, and the pde
                                #           loss curve was flat noise the whole run -- raising W_PDE
                                #           did nothing because T's only gradient path (the Laplacian
                                #           term) was apparently already near zero, not just weighted
                                #           too low. See STATUS.md eighth-run entry.
                                #   1e-4 -> Run 9 (2026-07-24): W_PDE held at Run 8's value; the
                                #           change was LAPLACIAN_SCALE (see below) + Tanh->SiLU.
                                #           T unfroze but RMSE regressed to 12.14m -- LAPLACIAN_SCALE
                                #           too strong; Run 10 backs it off. See STATUS.md.
                                # NOTE: data loss = MSE on NORMALIZED head (shrank by ~head_s^2 vs
                                # the old depth-MSE). 1e-4 here is roughly equivalent in relative
                                # weight to the old 1e-6 * head_s^2 rescale, but with the PDE now
                                # governing the correct physical quantity so a meaningful gradient
                                # can flow to S/T for the first time.
W_IC = 1.0
IC_WINDOW_DAYS = 60             # observations within this many days of the training set's earliest
                                # timestamp are treated as "initial condition" points. There is no clean
                                # gridded h(x,y,t=0) snapshot in the data (readings are sparse and
                                # irregular in time), so this is a practical stand-in, not a true IC.

# --- delta (residual-from-anchor) regularization (Run 21) ---------------------------------
# The Run 20 residual form makes the model predict h0hat + mlp(...), i.e. it STARTS at
# persistence and adds a learned deviation delta = mlp(...).  Locally re-scoring Run 20
# showed the net deviates from the anchor on 539/699 forecast wells and WINS on only 160
# (STATUS.md twentieth-run entry): the extrapolated train-window trend adds error instead
# of removing it, leaving PINN at 6.54m vs persistence 5.50m.  W_DELTA penalizes mean(delta^2)
# so the model only moves off the anchor when the data gives it a strong reason -- a soft
# "persistence is the floor" prior that shrinks useless trend extrapolation toward zero.
W_DELTA = 1e-2

# --- Run 22: trend features for the delta (STATUS.md Run 21, next-step 1) -----------------
# Run 21 left the forecast at 6.13m vs persistence 5.50m, and a local CPU check of the
# residual-from-anchor model at ~0 epochs (mlp ~ 0, i.e. the model IS persistence) scores
# 5.48m.  So 20,000 epochs of training currently makes the PRIMARY metric WORSE by ~0.65m:
# the learned delta is net-harmful, exactly as the per-well tally suggested (PINN won on
# 160/699 wells in Run 20).  W_DELTA shrank the damage; it could not make the delta useful,
# because the only time signal the MLP had was `that` -- and every forecast row sits OUTSIDE
# the training range of `that`, so the net was extrapolating a smooth function out of domain.
# These four flags each give the delta a signal that is valid OUTSIDE the training window, or
# stop the out-of-domain extrapolation outright.  Toggle ONE per run (CLAUDE.md), and log the
# result in STATUS.md.
USE_SEASONAL = False             # annual monsoon phase sin/cos, derived INSIDE the model
                                # `that` (see the Model cell) so it is periodic and therefore
                                # in-domain for any t, and so d/dt in pde_residual picks up the
                                # seasonal term through the chain rule automatically.  TN's
                                # water table has a strong annual cycle (recharge ~May-Dec,
                                # drawdown ~Jan-Apr) that a smooth MLP of raw t cannot express.
USE_VELOCITY = False            # per-well recent dh/dt before the cutoff, as an input feature
                                # alongside the h0 anchor.  Persistence knows each well's LAST
                                # LEVEL; this tells the net which WAY that level was moving --
                                # the first-order term persistence is missing.  Computed from
                                # train-window rows ONLY (same leak profile as h0), zero for
                                # held-out wells and collocation, and dropped out with the
                                # anchor so unanchored rows stay in-distribution.
VELOCITY_WINDOW_DAYS = 365      # OLS slope over this many days before the cutoff.  A year keeps
                                # the estimate from being a seasonal slope by construction.
VELOCITY_MIN_OBS = 5            # wells with fewer readings (or < 30d span) in the window get
VELOCITY_MIN_SPAN_DAYS = 30.0   # velocity 0, not a slope fitted through noise.
VELOCITY_CLIP_M_PER_DAY = 0.05  # +-18m/yr; beyond this it is a data artifact, not a trend.
USE_RAIN_ANOMALY = False        # trailing district rainfall anomaly, DESEASONALIZED (this
                                # year's last-W-days rainfall minus that district's day-of-year
                                # climatology, in sd units).  The seasonal features above give
                                # the model the average annual cycle; this gives it whether the
                                # CURRENT year is wetter or drier than that average -- the
                                # actual physical cause of a well deviating from its anchor.
                                # CAVEAT: unlike the velocity feature, this reads OBSERVED
                                # rainfall inside the forecast window, so the forecast number
                                # becomes "given the rainfall that fell", not a blind forecast.
                                # `Rhat` already had exactly this status (Run 17), so this adds
                                # no NEW leak -- but say so when reporting the number.
RAIN_ANOMALY_WINDOW_DAYS = 90   # ~one monsoon-response time for an unconfined aquifer.
CLAMP_T_TO_TRAIN = False        # Run 23 ran this ON: forecast 5.842m vs Run 22's 5.792m.  That
                                # 0.05m is far inside the 0.505m SEED band measured in Runs
                                # 26-30, so the honest reading is NO MEASURABLE EFFECT, not a
                                # loss.  Clamping removes
                                # out-of-domain extrapolation of `that` completely and by
                                # construction, so this run was a clean test of the "the net
                                # extrapolates fitted drift" hypothesis -- and refutes it.
                                # Reverted to False; do not re-run without a new reason.
                                # feed the MLP min(that, that_at_cutoff) instead of that.  The
                                # long-term drift the net learns over the training window is
                                # real signal INSIDE it and pure extrapolation outside it;
                                # clamping holds the drift at its last observed value and lets
                                # the seasonal/rainfall features carry the forecast window.
                                # Collocation is all t < cutoff, so the PDE (and T's gradient
                                # path) is untouched by this -- clamp'(z)=1 below the bound.

# --- PDE diffusion-term non-dimensionalization (added for Run 9, see STATUS.md) -----------
# A local smoke test (real cached elevations, few epochs, CPU) instrumented with the new
# |laplacian| diagnostic (see pde_residual/training-loop cells) showed the physical Laplacian
# is ~1e-11 EVEN AT EPOCH 0 -- not something that develops or vanishes over training, a fixed
# consequence of the chain-rule conversion head_s / x_s^2 (head varies ~200m; x spans Tamil
# Nadu's ~700km width in METERS, so x_s ~ 91,000m and x_s^2 ~ 8e9). T only receives gradient
# through `-T_d * laplacian`, so at this scale T's contribution to the residual is ~11 orders
# of magnitude smaller than R/Q (O(1-20)) -- invisible to gradient descent regardless of
# activation function or W_PDE, which is why Run 8's 100x W_PDE bump did nothing and why the
# Tanh -> SiLU swap alone (Model cell) is not expected to fix this by itself.
# LAPLACIAN_SCALE multiplies ONLY the diffusion term's contribution inside pde_residual (not
# T_raw's parametrization -- get_T() is unchanged) to make that gradient path visible again.
# Calibration history:
#   3e11 -> Run 9 (2026-07-24): sized so T_d*SCALE*|laplacian| ~ O(1-10), comparable to R/Q.
#           T unfroze (spread 0.44-2.73) BUT RMSE regressed to 12.14m (lost to the 12.07m
#           baseline) and spatial_maps.png's range collapsed to -20..35m, losing the correct
#           Western Ghats deep-water signal -- the diffusion term, being a function of the
#           network's OWN curvature, was strong enough to reshape the MLP's spatial weights and
#           overwrite the good Run 7/8 elevation fit. Too strong. See STATUS.md ninth-run entry.
#   1e10 -> current (Run 10): 30x smaller, sizing the diffusion term to ~O(0.4) -- a SUB-DOMINANT
#           regularizer (well under R/Q's O(1-20)) instead of a competing objective. A local
#           smoke test confirms T still gets real gradient here (|T_raw.grad| ~1.7e-3 at 30
#           epochs, ~11 orders of magnitude above the ~1e-14 frozen-T noise floor), just far
#           more gently, so the network's data-driven fit should survive. If this STILL regresses
#           vs Run 7 (11.78m), see STATUS.md: frozen-T Run 7 is the accuracy reference and
#           learning T at all is a deliberate accuracy-vs-physics trade, not a value to keep tuning.
# IMPORTANT CAVEAT: R and Q are documented proxies without true SI calibration (Q is a
# unitless [0,1] index, not m3/day -- DATA_DICTIONARY.md limitation 4), so there was never a
# path to recovering literal m2/day transmissivity from this residual as posed. LAPLACIAN_SCALE
# does not fix that -- it only makes T learnable AT ALL (real cross-district variation) instead
# of frozen. Treat learned_S_T_per_district.csv's T column as a RELATIVE quantity (compare
# districts to each other) after this change, not literal transmissivity, unless R/Q are
# recalibrated to true SI units in a future revision.
LAPLACIAN_SCALE = 1e10

# --- training ---------------------------------------------------------
HELD_OUT_WELL_FRACTION = 0.15
# --- Run 12: FORECAST split (see STATUS.md twelfth-run entry) --------------------------
# Runs 7-11 were scored only on held-out WELLS. Local diagnostics showed that task is at its
# feature ceiling: kNN(x,y) on train well means scores 11.62m vs the PINN's best 11.51m, because
# 73.6% of h's variance is BETWEEN-well and is not spatially smooth -- no smooth f(x,y,elev) can
# interpolate it at an unmonitored location. Splitting on TIME instead gives real headroom
# (trivial baselines 5.55-5.71m, oracle floor 3.73m) and matches the actual use case: predicting
# groundwater at monitored wells going forward. Both tasks are now reported; forecast is primary.
TEMPORAL_CUTOFF_QUANTILE = 0.80   # train on t below this quantile of observation time, evaluate
                                  # at/above it. 0.80 -> t~20057d (~2024.9), a ~1.2-year horizon
                                  # over ~810 wells / ~906k rows.
USE_TEMPORAL_WEIGHTS = False    # ADOPTED (Runs 25-30, 3 seeds/cell).  Run 25.  Batches were sampled with weights inverse to rows
                                # per calendar year, so each of 53 years contributed equally.
                                # That was added when the ONLY metric was held-out wells over
                                # all time.  The protocol has been forecast-primary since Run
                                # 12, and measured on the real split (STATUS.md Run 24):
                                #   2021+ telemetry  = 69.2% of train rows ->  7.5% of gradient
                                #   wells the forecast scores = 76.8% of rows -> 17.5%
                                # i.e. ~92% of the data gradient went to an era and a well
                                # population that NO reported metric measures.  False = plain
                                # uniform sampling, which puts the gradient where the
                                # evaluation is.  The old behaviour is still one flag away;
                                # if the spatial (secondary) metric collapses, that is the
                                # cost of this trade and should be logged as such.
BATCH_SIZE_DATA = 4096
BATCH_SIZE_COLLOC = 4096
BATCH_SIZE_IC = 512
N_EPOCHS_ADAM = 20000
ADAM_LR = 1e-3
WEIGHT_DECAY = 3e-4             # bumped from 1e-4 alongside the smaller network above -- same
                                # memorization concern ((x,y) uniquely identifies a well, so the net
                                # can just memorize the ~2,655 training wells: train RMSE ~5m vs
                                # held-out 15.6m on the very first run).
LR_STEP_SIZE = 5000
LR_GAMMA = 0.5
PRINT_EVERY = 500
RUN_LBFGS_FINETUNE = False   # Run 22: Run 21's L-BFGS stepped Adam's data loss back up
                              # (0.26 ~ 5.3m depth -> 0.52 ~ 7.4m). The fine-tune fights
                              # the weighted objective; keep the better Adam fit.
LBFGS_MAX_ITER = 500

SEED = 1                        # optimization seed -- weight init and batch draws.
                                # Training is deterministic (Run 25b repeated Run 25 bit-for-bit),
                                # so perturbing this is the only way to measure the project's real
                                # metric resolution.  Sweep against Run 25's SEED=0.
SPLIT_SEED = 0                  # held-out-well draw draw
                                # SEED to 1 and silently redrew the evaluation set too -- every
                                # baseline moved (persistence 5.500 -> 5.686, kNN 11.573 -> 9.861,
                                # district mean 12.013 -> 9.256), so that run measured a different
                                # benchmark, not optimization noise.  Keep this pinned at 0: it is
                                # the split every run from 7 on was scored against, and changing it
                                # invalidates comparison with the whole log above.

# ══ ACCURACY IMPROVEMENT STEPS (Run 32+) ════════════════════════════════════

# Step 1: learnable delta scale alpha. Run 31 post-hoc sweep found alpha=0.3 optimal.
# log_alpha initialised at log(0.3); softplus -> ~0.30 at step 0.
# False = Run 31 baseline (no scaling).
USE_ALPHA_SCALE = True

# Step 2: 90-day rolling anchor. Pre-cutoff diagnostic: 90d-mean RMSE=5.406m vs 5.500m last-obs.
# H0_ANCHOR_DAYS > 0 uses mean of that many days before cutoff per well; 0 = last-obs.
H0_ANCHOR_DAYS = 90

# Step 3: W_PDE=0 ablation flag (run as a separate Kaggle push, not here).
# Uncomment the line below and comment the current W_PDE line for the ablation run.
# W_PDE = 0.0

# Step 4: Antecedent Precipitation Index. API_k(t) = sum R(t-tau)*exp(-tau/k).
# Captures infiltration lag between rainfall and water-table response (4-8 weeks in TN).
USE_API = True
API_DECAY_DAYS = [30, 90]   # e-folding decay constants in days

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load data

Raises immediately if either file is missing or malformed -- never falls back to synthetic
placeholder data (see CLAUDE.md).

In [ ]:
REQUIRED_OBS_COLS = {"x", "y", "t", "district", "h", "R", "Q"}
REQUIRED_COLLOC_COLS = {"x", "y", "t", "district", "R", "Q"}

if not os.path.exists(OBSERVATIONS_PATH):
    raise FileNotFoundError(
        f"{OBSERVATIONS_PATH} not found. This notebook requires the Phase 1 output; "
        f"set DATA_DIR to wherever it's mounted. Refusing to generate synthetic data instead."
    )
if not os.path.exists(COLLOCATION_PATH):
    raise FileNotFoundError(
        f"{COLLOCATION_PATH} not found. This notebook requires the Phase 1 output; "
        f"set DATA_DIR to wherever it's mounted. Refusing to generate synthetic data instead."
    )

observations = pd.read_csv(OBSERVATIONS_PATH)
collocation = pd.read_csv(COLLOCATION_PATH)

missing_obs_cols = REQUIRED_OBS_COLS - set(observations.columns)
missing_colloc_cols = REQUIRED_COLLOC_COLS - set(collocation.columns)
if missing_obs_cols:
    raise ValueError(f"observations.csv is missing required column(s): {missing_obs_cols}")
if missing_colloc_cols:
    raise ValueError(f"collocation.csv is missing required column(s): {missing_colloc_cols}")

print(f"observations: {len(observations):,} rows, {observations['district'].nunique()} districts, "
      f"t range [{observations['t'].min():.1f}, {observations['t'].max():.1f}]")
print(f"collocation: {len(collocation):,} rows, {collocation['district'].nunique()} districts")
print(f"observations.h: min={observations['h'].min():.2f} max={observations['h'].max():.2f} "
      f"(depth to water table, m, positive = deeper -- see DATA_DICTIONARY.md)")


## Recharge fallback for no-humidity districts

12 of 38 districts have no humidity data → `R` is NaN in both `observations.csv` and
`collocation.csv` for those districts → they contribute **zero rows to the PDE loss**
and `S`/`T` cannot receive PDE gradients regardless of `W_PDE`.

Fix applied here (not in Phase 1 to avoid regenerating the 368 MB `observations.csv`):
load `rainfall_daily.csv` from the same `DATA_DIR`, aggregate to monthly district means,
and fill NaN `R` values in `collocation` using the simplified formula

    R_fallback = max(rainfall_mm - ET_K * 50, 0)

where `ET_K * 50` is the ET proxy at 50 % RH — the middle of the plausible Tamil Nadu
range — used in place of the missing humidity reading.  This is a rough approximation,
explicitly logged below; it is strictly better than leaving `R = NaN` (= no PDE signal)
for these districts.

For Mayiladuthurai and Perambalur (no rainfall either), the state-wide monthly mean
`R` from the districts that do have a value is used as a last resort.

In [ ]:
# --- R fallback: rainfall-only proxy for districts with no humidity -------------------------
# 10 of the 12 no-humidity districts have rainfall coverage; 2 (Mayiladuthurai, Perambalur)
# have neither.  Any NaN R remaining after this block uses the state-wide monthly mean.
RAINFALL_PATH = os.path.join(DATA_DIR, "rainfall_daily.csv")
ET_K = 0.1           # must match build_recharge.py
ET_MID_HUMIDITY = 50 # proxy: assume 50% RH where humidity is unknown

EPOCH_TS = pd.Timestamp("1970-01-01")

if os.path.exists(RAINFALL_PATH):
    rf = pd.read_csv(RAINFALL_PATH, parse_dates=["date"])
    # aggregate to district-month mean rainfall
    rf["month_ts"] = rf["date"].values.astype("datetime64[M]").astype("datetime64[ns]")
    rf["month_t"] = (rf["month_ts"] - EPOCH_TS).dt.total_seconds() / 86400.0
    rf_monthly = (rf.groupby(["canonical_district", "month_t"], as_index=False)["rainfall_mm"].mean()
                    .rename(columns={"canonical_district": "district"}))
    # R_fallback = max(rainfall - ET_proxy_at_mid_humidity, 0)
    rf_monthly["R_fallback"] = (rf_monthly["rainfall_mm"] - ET_K * (100 - ET_MID_HUMIDITY)).clip(lower=0)

    # merge onto collocation -- only fills where collocation's t matches a month_t in rainfall
    before_fill = collocation["R"].isna().sum()
    collocation = collocation.merge(rf_monthly[["district","month_t","R_fallback"]],
                                    left_on=["district","t"], right_on=["district","month_t"],
                                    how="left").drop(columns=["month_t"])
    collocation["R"] = collocation["R"].fillna(collocation["R_fallback"])
    collocation = collocation.drop(columns=["R_fallback"])
    after_fill = collocation["R"].isna().sum()
    print(f"R fallback (rainfall-only): filled {before_fill - after_fill:,} of {before_fill:,} NaN R rows "
          f"in collocation using rainfall_daily.csv")
else:
    print(f"WARNING: {RAINFALL_PATH} not found -- rainfall R-fallback skipped. "
          f"12 no-humidity districts will still contribute zero PDE rows.")

# Last resort: any remaining NaN R gets the state-wide monthly mean R from known districts
still_nan = collocation["R"].isna().sum()
if still_nan > 0:
    state_monthly_mean = (collocation.dropna(subset=["R"])
                          .groupby("t")["R"].mean().rename("R_state_mean"))
    collocation = collocation.join(state_monthly_mean, on="t")
    collocation["R"] = collocation["R"].fillna(collocation["R_state_mean"])
    collocation = collocation.drop(columns=["R_state_mean"])
    print(f"State-monthly-mean fallback: filled remaining {still_nan:,} R NaNs "
          f"(Mayiladuthurai, Perambalur, or any unmatched rows)")

n_nan_after = collocation["R"].isna().sum()
print(f"Collocation R NaNs after all fallbacks: {n_nan_after} "
      f"({'none -- all districts now have PDE coverage' if n_nan_after == 0 else 'some remain, check above'})")


## Elevation lookup (depth -> head)

`h` in `observations.csv` is depth-to-water below the local ground surface, but the PDE governs
*hydraulic head* -- two wells with the same head can have very different depth-to-water if their
ground elevations differ (STATUS.md issue 2/4). No DEM ships in `data/raw/`, so this cell fetches
ground elevation at runtime for every unique `(lon, lat)` point across both `observations` and the
`collocation` grid (~4,000 points total), from the free [Open Topo Data](https://www.opentopodata.org/)
SRTM 30m API -- no key required, but **this needs internet access**: on Kaggle, turn on
`Settings > Internet` for the notebook, or this cell raises immediately (no synthetic fallback, per
CLAUDE.md).

Results are cached to `ELEVATION_CACHE_PATH` so a kernel restart doesn't re-fetch from scratch.
Points the API can't resolve (e.g. large water bodies) come back `null`; those rows are dropped
from both frames (count printed), not zero-filled.

**Design note (post-mortem on the first head-conversion run)**: `elev` is fed into the network as
an explicit **input** feature (see Model section below), not only used to build the `head` target.
The first attempt used it target-side only -- Tamil Nadu's elevation spans ~2,400m (coastal plains
to the Nilgiris), so `head = elev - h` inherited nearly all of that range, and the network had no
way to predict it since it never saw elevation, only `(x, y, t, district)`. It fit training wells
fine but blew up on held-out wells in high-relief districts (RMSE 130m vs. the 12m baseline) --
it was forced to blindly reconstruct terrain instead of just fitting the (much smoother) water
table. Feeding `elev` in directly lets the network read the terrain off instead of memorizing it.


In [ ]:
def xy_key(df):
    return df["x"].round(5).astype(str) + "_" + df["y"].round(5).astype(str)

observations["xy_key"] = xy_key(observations)
collocation["xy_key"] = xy_key(collocation)

all_xy = (pd.concat([observations[["xy_key", "x", "y"]], collocation[["xy_key", "x", "y"]]])
          .drop_duplicates(subset="xy_key").reset_index(drop=True))
print(f"{len(all_xy):,} unique (lon, lat) points need elevation (observations + collocation grid)")

if os.path.exists(ELEVATION_CACHE_PATH):
    elev_cache = pd.read_csv(ELEVATION_CACHE_PATH)
    print(f"loaded {len(elev_cache):,} cached elevations from {ELEVATION_CACHE_PATH}")
else:
    elev_cache = None   # built fresh below from fetched_rows -- NOT pd.DataFrame(columns=[...]),
                         # which has no data to infer dtype from and leaves "elev" as `object`;
                         # concatenating fetched (float) rows onto that silently poisons the
                         # whole column to `object`, which then breaks every downstream numeric
                         # op (head, elevhat, headhat) and torch.tensor() with a confusing error.
                         # Only shows up on a truly fresh cache -- a reload from CSV self-heals,
                         # since read_csv infers proper dtypes -- so this only bites the first run.

to_fetch = all_xy if elev_cache is None else all_xy[~all_xy["xy_key"].isin(elev_cache["xy_key"])].reset_index(drop=True)
print(f"{len(to_fetch):,} points not yet cached, fetching from {ELEVATION_API_URL}")

fetched_rows = []
for start in range(0, len(to_fetch), ELEVATION_BATCH_SIZE):
    chunk = to_fetch.iloc[start:start + ELEVATION_BATCH_SIZE]
    locations = "|".join(f"{lat:.5f},{lon:.5f}" for lon, lat in zip(chunk["x"], chunk["y"]))

    for attempt in range(1, ELEVATION_MAX_RETRIES + 1):
        try:
            resp = requests.get(ELEVATION_API_URL, params={"locations": locations}, timeout=30)
            resp.raise_for_status()
            results = resp.json()["results"]
            break
        except (requests.RequestException, ValueError) as e:
            if attempt == ELEVATION_MAX_RETRIES:
                raise RuntimeError(
                    f"elevation API failed after {ELEVATION_MAX_RETRIES} attempts on batch "
                    f"{start}-{start + len(chunk)}: {e}. If this is Kaggle, check that "
                    f"Settings > Internet is turned on for this notebook."
                ) from e
            time.sleep(2 ** attempt)

    if len(results) != len(chunk):
        raise ValueError(f"elevation API returned {len(results)} results for {len(chunk)} points")
    for key, r in zip(chunk["xy_key"], results):
        fetched_rows.append({"xy_key": key, "elev": r["elevation"]})

    time.sleep(ELEVATION_RATE_LIMIT_S)
    done = min(start + ELEVATION_BATCH_SIZE, len(to_fetch))
    if (start // ELEVATION_BATCH_SIZE) % 5 == 0 or done == len(to_fetch):
        print(f"  fetched {done:,}/{len(to_fetch):,}")

if fetched_rows:
    new_elev = pd.DataFrame(fetched_rows)
    elev_cache = pd.concat([elev_cache, new_elev], ignore_index=True) if elev_cache is not None else new_elev
    elev_cache.to_csv(ELEVATION_CACHE_PATH, index=False)
    print(f"cached {len(fetched_rows):,} new elevations to {ELEVATION_CACHE_PATH}")

# belt-and-braces: guarantee numeric dtype regardless of how elev_cache was assembled above
elev_cache["elev"] = pd.to_numeric(elev_cache["elev"], errors="coerce")

n_null = elev_cache["elev"].isna().sum()
if n_null:
    print(f"*** {n_null:,} of {len(elev_cache):,} points returned null elevation "
          f"(likely off SRTM coverage, e.g. large water bodies) -- DROPPED, not zero-filled ***")

observations = observations.merge(elev_cache, on="xy_key", how="left")
collocation = collocation.merge(elev_cache, on="xy_key", how="left")

# elev is now used as a MODEL INPUT everywhere below (not just the target conversion), so both
# frames need it non-null before any tensors are built -- dropped here, once, not deferred to
# individual cells.
n_obs_before, n_colloc_before = len(observations), len(collocation)
observations = observations.dropna(subset=["elev"]).reset_index(drop=True)
collocation = collocation.dropna(subset=["elev"]).reset_index(drop=True)
print(f"observations: dropped {n_obs_before - len(observations):,} of {n_obs_before:,} rows with "
      f"no elevation match; {len(observations):,} remain")
print(f"collocation: dropped {n_colloc_before - len(collocation):,} of {n_colloc_before:,} rows "
      f"with no elevation match; {len(collocation):,} remain")

# depth-to-water -> hydraulic head estimate (meters, same vertical datum as elev)
observations["head"] = observations["elev"] - observations["h"]
print(f"head: mean={observations['head'].mean():.2f}m std={observations['head'].std():.2f}m "
      f"(elev mean={observations['elev'].mean():.2f}m, h mean={observations['h'].mean():.2f}m)")


## Preprocessing

- Drop observations outside `[H_MIN, H_MAX]` -- Phase 1's `|h| >= 200` sentinel cut is too
  loose (see config cell). Counts are printed; nothing is silently filled or clipped. Applied
  to raw depth `h`, ahead of the head conversion above -- this is a bad-reading filter, not a
  head/depth judgment call.
- Convert lon/lat to local meters (equirectangular approximation around the domain's mean
  latitude) -- training directly in degree-space would make the PDE's `S`/`T` parameters
  physically meaningless, since 1 degree of longitude and 1 degree of latitude are not the
  same distance.
- Normalize (x_m, y_m, t_day, elev_m) to roughly [-1, 1] for network stability; the scale
  constants are kept so PDE derivatives can be converted back to physical units via the chain
  rule. `elev` is normalized the same way as the spatial inputs since it's fed into the network
  as an input feature now (see elevation lookup cell), not just used to build the target.
- **Normalize the target `head` (`head_c`, `head_s`)**: the network predicts standardized
  `headhat`, not raw meters, and not raw depth `h` -- see the elevation cell above for why
  head is the physically correct target for the PDE. `pde_residual()` and the evaluation cell
  convert back to physical units (and, at the two display boundaries, back to depth-to-water
  via `elev`) so nothing downstream is silently in the wrong unit.
- Build a district -> integer index map from the union of both files (sorted, so it's
  reproducible run to run).
- Collocation rows with no `R` (see DATA_DICTIONARY.md -- 12 of 38 districts have no humidity
  coverage) are dropped for the PDE loss: the residual is undefined without a real R value, and
  filling it with 0 would silently assert "no recharge" for districts that were simply never
  measured.
- Held-out split is by **well** (unique rounded (x, y) location), not by row, to avoid a
  station's readings leaking between train and test across time.


In [ ]:
# --- h range filter (see config cell for why the Phase 1 cut isn't enough) ---
n_before = len(observations)
out_of_range = ~observations["h"].between(H_MIN, H_MAX)
print(f"h range filter [{H_MIN}, {H_MAX}]: dropping {out_of_range.sum():,} of {n_before:,} rows "
      f"({100 * out_of_range.mean():.2f}%)")
print(f"  below {H_MIN}: {(observations['h'] < H_MIN).sum():,} rows   "
      f"above {H_MAX}: {(observations['h'] > H_MAX).sum():,} rows")
observations = observations[~out_of_range].reset_index(drop=True)
print(f"  kept {len(observations):,} rows; h now spans "
      f"[{observations['h'].min():.2f}, {observations['h'].max():.2f}], std {observations['h'].std():.3f}m")

REF_LAT = observations["y"].mean()
REF_LON = observations["x"].mean()
M_PER_DEG_LAT = 110_540.0
M_PER_DEG_LON = 111_320.0 * math.cos(math.radians(REF_LAT))

def to_local_meters(lon, lat):
    x_m = (lon - REF_LON) * M_PER_DEG_LON
    y_m = (lat - REF_LAT) * M_PER_DEG_LAT
    return x_m, y_m

for df in (observations, collocation):
    df["x_m"], df["y_m"] = to_local_meters(df["x"], df["y"])

# district -> index, built from the union of both files so an index is never missing
all_districts = sorted(set(observations["district"]) | set(collocation["district"]))
district_to_idx = {d: i for i, d in enumerate(all_districts)}
NUM_DISTRICTS = len(all_districts)
for df in (observations, collocation):
    df["district_idx"] = df["district"].map(district_to_idx)
print(f"{NUM_DISTRICTS} districts indexed")

# normalization constants from the training-relevant range (observations define the domain)
x_c, x_s = observations["x_m"].mean(), observations["x_m"].std()
y_c, y_s = observations["y_m"].mean(), observations["y_m"].std()
t_c, t_s = observations["t"].mean(), observations["t"].std()
elev_c, elev_s = observations["elev"].mean(), observations["elev"].std()
head_c, head_s = observations["head"].mean(), observations["head"].std()
h_c, h_s = observations["h"].mean(), observations["h"].std()
# R/Q normalization -- R is NaN for ~80% of rows (pre-2021, no-humidity districts);
# fill with 0 (no-recharge baseline) for those rows.
R_c = observations["R"].fillna(0.0).mean()
R_s = observations["R"].fillna(0.0).std()
Q_c = observations["Q"].mean()
Q_s = observations["Q"].std()
print(f"x_s={x_s:.1f}m  y_s={y_s:.1f}m  t_s={t_s:.1f}days  elev_c={elev_c:.1f}m  elev_s={elev_s:.1f}m  "
      f"head_c={head_c:.3f}m  head_s={head_s:.3f}m  h_c={h_c:.3f}m  h_s={h_s:.3f}m")
print("  (chain-rule scale constants -- pde_residual and the eval cell both convert back to physical units)")

# --- Run 11: the network now predicts normalized DEPTH h, not normalized HEAD -------------------
# Measured on the real data (4.53M rows after the elev merge and the h filter):
#   corr(head, elev) = 0.99866, and elev alone accounts for ~100% of head's variance.
# head = elev - h, and `elev` is ALREADY A NETWORK INPUT (elevhat), so predicting head let the
# network score well by echoing an input. The quantity we actually care about, h, is only
# h_s/head_s = 5.2% of the target's std and 0.27% of its VARIANCE -- so MSE on normalized head
# barely registers depth errors at all. That is the mechanism behind the saturating band in
# every pred_vs_actual.png from Run 7 on: the loss was nearly indifferent to it.
# Predicting h directly makes the loss 373x more sensitive to the same physical error.
# This costs NOTHING physically -- see pde_residual, where the residual is algebraically the
# same quantity as before.
print(f"target: normalized h (depth). head_s/h_s = {head_s / h_s:.1f}x, "
      f"variance ratio {(head_s / h_s) ** 2:.0f}x -- see comment above")

# --- Run 22: SEASONAL PHASE, as constants the MODEL applies to `that` ----------------------
# sin/cos of the annual cycle are NOT precomputed as dataframe columns.  They are derived
# inside PINN.forward from `that` itself, using the two constants below:
#     t_days = that * t_s + t_c   ->   phase = 2*pi*t_days/365.25 = SEAS_W*that + SEAS_B
# Two reasons this must live in the model rather than in a column:
#  1. CORRECTNESS.  pde_residual differentiates the network w.r.t. `that` to get dh/dt.  If the
#     seasonal terms arrived as separate constant inputs, autograd would never see that they
#     are functions of t, and dh/dt would silently omit the entire seasonal contribution --
#     i.e. the PDE would be fed a time derivative that is wrong by the largest term in it.
#     Deriving them from `that` puts them in the graph, and the chain rule handles the rest.
#  2. No `mod` is needed: sin/cos of 2*pi*t/365.25 is already periodic, so this is smooth
#     everywhere (a doy = t mod 365.25 column has a derivative discontinuity every new year).
SEAS_W = 2 * np.pi * t_s / 365.25
SEAS_B = 2 * np.pi * t_c / 365.25
print(f"seasonal phase constants: phase = {SEAS_W:.4f}*that + {SEAS_B:.4f} "
      f"({'ON' if USE_SEASONAL else 'OFF'})")

for df in (observations, collocation):
    df["xhat"] = (df["x_m"] - x_c) / x_s
    df["yhat"] = (df["y_m"] - y_c) / y_s
    df["that"] = (df["t"] - t_c) / t_s
    df["elevhat"] = (df["elev"] - elev_c) / elev_s
    df["Rhat"] = (df["R"].fillna(R_c) - R_c) / R_s
    df["Qhat"] = (df["Q"] - Q_c) / Q_s
observations["hhat"] = (observations["h"] - h_c) / h_s   # network predicts this, not raw meters

# well = unique rounded (lon, lat) location, since observations.csv has no station-id column
observations["well_id"] = (observations["x"].round(5).astype(str) + "_" + observations["y"].round(5).astype(str))
unique_wells = observations["well_id"].unique()
rng = np.random.default_rng(SPLIT_SEED)   # Run 27: NOT SEED -- see config.  SPLIT_SEED=0
                                          # reproduces the Runs 7-25 well holdout exactly.
held_out_wells = set(rng.choice(unique_wells, size=int(len(unique_wells) * HELD_OUT_WELL_FRACTION), replace=False))
is_held_out = observations["well_id"].isin(held_out_wells)

# Run 12: three-way split. The well holdout above is UNCHANGED from Runs 7-11 (same SEED, same
# rng draw order) so the spatial metric stays directly comparable to every prior run.
t_cutoff = observations["t"].quantile(TEMPORAL_CUTOFF_QUANTILE)

train_df = observations[~is_held_out & (observations["t"] < t_cutoff)].reset_index(drop=True)

# FORECAST test set (primary): wells the model trained on, at times it never saw. The
# "well appears in train_df" filter is what makes this a forecast rather than a cold start --
# it drops wells whose records end before the cutoff (mostly pre-2021 manual stations), which
# would otherwise be spatial generalization smuggled into the forecast number.
trained_wells = set(train_df["well_id"])
test_forecast_df = observations[
    ~is_held_out & (observations["t"] >= t_cutoff) & observations["well_id"].isin(trained_wells)
].reset_index(drop=True)

# SPATIAL test set (secondary): held-out wells, all t. Identical to Runs 7-11's `test_df`.
test_spatial_df = observations[is_held_out].reset_index(drop=True)

print(f"{len(unique_wells)} unique wells -> {len(held_out_wells)} held out for the spatial task")
print(f"t_cutoff = {t_cutoff:.0f}d (~{1970 + t_cutoff / 365.25:.1f}), "
      f"quantile {TEMPORAL_CUTOFF_QUANTILE}")
print(f"  train           {len(train_df):>10,} rows  {train_df['well_id'].nunique():>5} wells  "
      f"(kept wells, t < cutoff)")
print(f"  test/forecast   {len(test_forecast_df):>10,} rows  "
      f"{test_forecast_df['well_id'].nunique():>5} wells  (kept wells, t >= cutoff, seen in train)")
print(f"  test/spatial    {len(test_spatial_df):>10,} rows  "
      f"{test_spatial_df['well_id'].nunique():>5} wells  (held-out wells, all t)")
n_dropped = len(observations) - len(train_df) - len(test_forecast_df) - len(test_spatial_df)
print(f"  unused          {n_dropped:>10,} rows  (kept wells past the cutoff whose well never "
      f"appears before it -- excluded, not folded into either test set)")

ic_cutoff = train_df["t"].min() + IC_WINDOW_DAYS
ic_df = train_df[train_df["t"] <= ic_cutoff]
print(f"{len(ic_df):,} rows within {IC_WINDOW_DAYS} days of the training set's earliest "
      f"timestamp, used as the IC term")

# --- Step 2 / Run 19: per-well baseline anchor feature --------------------------------
# Step 2 (H0_ANCHOR_DAYS=90): pre-cutoff 90d-mean anchor baseline = 5.406m RMSE
# vs 5.500m for last-obs (Run 22 prep diagnostic). Falls back to last-obs per well
# if no reading exists in the window. 0 = last-obs (all runs up to 31).
well_sorted = train_df.sort_values("t")
well_last_tr = well_sorted.groupby("well_id")["h"].last()       # last-obs (fallback)
if H0_ANCHOR_DAYS > 0:
    anchor_win = well_sorted[well_sorted["t"] >= t_cutoff - H0_ANCHOR_DAYS]
    h0_rolling = anchor_win.groupby("well_id")["h"].mean()      # rolling mean
    h0_anchor = h0_rolling.combine_first(well_last_tr)          # fallback where window empty
    n_windowed = int(h0_rolling.notna().sum())
    print(f"Step 2 anchor: H0_ANCHOR_DAYS={H0_ANCHOR_DAYS} -> "
          f"{n_windowed:,} wells use rolling mean; "
          f"{len(well_last_tr) - n_windowed:,} fall back to last-obs")
    fc_h0_rmse_val = float(
        ((test_forecast_df["well_id"].map(h0_anchor).fillna(h_c)
          - test_forecast_df["h"]) ** 2).mean() ** 0.5)
    print(f"  {H0_ANCHOR_DAYS}d-mean anchor RMSE on forecast set: {fc_h0_rmse_val:.3f}m "
          f"(expected ~5.406m; last-obs is 5.500m)")
else:
    h0_anchor = well_last_tr
    print(f"Step 2 anchor: H0_ANCHOR_DAYS=0 -> last-obs (Run 31 default, 5.500m baseline)")
for df in (train_df, test_forecast_df, test_spatial_df, ic_df):
    df["h0"] = df["well_id"].map(h0_anchor)
    df["h0hat"] = (df["h0"].fillna(h_c) - h_c) / h_s
print(f"anchor: {train_df['well_id'].nunique():,} train; "
      f"{test_forecast_df['well_id'].nunique():,} forecast; "
      f"{len(test_spatial_df):,} spatial/held-out rows get h0hat=0")

# --- Run 22: per-well VELOCITY feature (train-window only) ---------------------------------
# The anchor h0 tells the net WHERE each well was at the cutoff.  This tells it which way the
# well was MOVING -- the first-order term persistence has no access to.  Fitted as an OLS slope
# of h on t over the last VELOCITY_WINDOW_DAYS of the TRAIN window, so it cannot leak: it uses
# strictly the same rows the anchor does.  Wells with too few readings or too short a span get
# 0 (= "no trend information"), which is the same value held-out wells and collocation get, so
# 0 is a well-populated, in-distribution value rather than a rare sentinel.
if USE_VELOCITY:
    vw = train_df[train_df["t"] >= t_cutoff - VELOCITY_WINDOW_DAYS]
    gv = vw.groupby("well_id")
    dt_ = vw["t"] - gv["t"].transform("mean")
    dh_ = vw["h"] - gv["h"].transform("mean")
    num = (dt_ * dh_).groupby(vw["well_id"]).sum()
    den = (dt_ ** 2).groupby(vw["well_id"]).sum()
    n_obs = gv["t"].size()
    span = gv["t"].max() - gv["t"].min()
    ok = (n_obs >= VELOCITY_MIN_OBS) & (span >= VELOCITY_MIN_SPAN_DAYS) & (den > 0)
    vel = (num / den).where(ok).clip(-VELOCITY_CLIP_M_PER_DAY, VELOCITY_CLIP_M_PER_DAY)
    vel_s = float(vel.std())
    if not np.isfinite(vel_s) or vel_s == 0.0:
        raise ValueError("velocity feature has zero/NaN spread -- check the window, not a fallback")
    n_clipped = int(((num / den).where(ok).abs() > VELOCITY_CLIP_M_PER_DAY).sum())
    print(f"velocity feature: fitted over the last {VELOCITY_WINDOW_DAYS}d of the train window; "
          f"{int(ok.sum()):,} of {len(ok):,} train wells qualify "
          f"(>= {VELOCITY_MIN_OBS} obs, >= {VELOCITY_MIN_SPAN_DAYS:.0f}d span), "
          f"{len(ok) - int(ok.sum()):,} get 0; {n_clipped:,} clipped at "
          f"+-{VELOCITY_CLIP_M_PER_DAY} m/day")
    print(f"  slope m/day: p05={vel.quantile(0.05):+.5f} p50={vel.quantile(0.50):+.5f} "
          f"p95={vel.quantile(0.95):+.5f}  (x365 -> {vel.quantile(0.05) * 365:+.2f} / "
          f"{vel.quantile(0.50) * 365:+.2f} / {vel.quantile(0.95) * 365:+.2f} m/yr)  sd={vel_s:.5f}")
    for df in (train_df, test_forecast_df, test_spatial_df, ic_df):
        df["velhat"] = df["well_id"].map(vel).fillna(0.0) / vel_s
    # held-out wells: map() yields NaN -> 0, same convention as h0hat.  Verify, don't assume:
    print(f"  spatial (held-out) rows with velhat != 0: "
          f"{int((test_spatial_df['velhat'] != 0).sum()):,} (expected 0)")

# --- Run 22: trailing DESEASONALIZED rainfall anomaly, per district ------------------------
# R itself is NaN on ~80% of observation rows (no humidity before 2021 / in 12 districts), so it
# cannot carry the forecast window on its own.  Rainfall is far better covered (34-36 districts
# from 2021 on, which is where the cutoff and the whole forecast window live).  The feature is:
#   anom(d, t) = [ mean rainfall over (t-W, t] ]  -  [ that district's day-of-year climatology
#                                                     of the same statistic ]   / sd
# so it is 0 in a normal year and positive in a wetter-than-usual one.  The climatology and the
# sd are computed on TRAIN-WINDOW days only -- forecast-window rainfall never defines the
# baseline it is measured against.  Missing days are left NaN and excluded from the rolling
# mean (min_periods), never filled with 0: "no gauge record" is not "no rain".
if USE_RAIN_ANOMALY:
    RAIN_PATH = os.path.join(DATA_DIR, "rainfall_daily.csv")
    if not os.path.exists(RAIN_PATH):
        raise FileNotFoundError(
            f"{RAIN_PATH} not found but USE_RAIN_ANOMALY=True. Mount rainfall_daily.csv "
            f"alongside observations.csv, or set USE_RAIN_ANOMALY=False. Not zero-filling."
        )
    rfa = pd.read_csv(RAIN_PATH, parse_dates=["date"])
    rfa = (rfa.groupby(["canonical_district", "date"], as_index=False)["rainfall_mm"].mean()
              .rename(columns={"canonical_district": "district"}))
    rfa["day"] = (rfa["date"] - pd.Timestamp("1970-01-01")).dt.days
    # dense daily grid per district so the rolling window is a real W-day window, not W records
    piv = rfa.pivot_table(index="day", columns="district", values="rainfall_mm", aggfunc="mean")
    piv = piv.reindex(range(int(piv.index.min()), int(piv.index.max()) + 1))
    roll = piv.rolling(RAIN_ANOMALY_WINDOW_DAYS, min_periods=max(3, RAIN_ANOMALY_WINDOW_DAYS // 3)).mean()
    roll_long = roll.reset_index().melt(id_vars="day", var_name="district", value_name="roll")
    roll_long["doy"] = (roll_long["day"] % 365.25).astype(int)
    train_mask = roll_long["day"] < t_cutoff
    clim = (roll_long[train_mask].groupby(["district", "doy"])["roll"].mean().rename("clim"))
    roll_long = roll_long.merge(clim, on=["district", "doy"], how="left")
    roll_long["anom"] = roll_long["roll"] - roll_long["clim"]
    anom_s = float(roll_long.loc[train_mask, "anom"].std()) or 1.0
    roll_long["ranomhat"] = roll_long["anom"] / anom_s
    ranom = roll_long.set_index(["district", "day"])["ranomhat"]
    n_cov = int(roll_long["ranomhat"].notna().sum())
    print(f"rain anomaly: {RAIN_ANOMALY_WINDOW_DAYS}d trailing mean, deseasonalized by "
          f"district x day-of-year climatology fitted on t < cutoff only; sd={anom_s:.3f} mm/day; "
          f"{n_cov:,} of {len(roll_long):,} district-days covered "
          f"({roll_long['district'].nunique()} districts)")
    for name, df in [("train", train_df), ("forecast", test_forecast_df),
                     ("spatial", test_spatial_df), ("ic", ic_df), ("colloc", collocation)]:
        key = pd.MultiIndex.from_arrays([df["district"], np.floor(df["t"]).astype(int)])
        vals = ranom.reindex(key).to_numpy()
        df["ranomhat"] = np.nan_to_num(vals, nan=0.0)
        print(f"  {name:9s} {np.isfinite(vals).mean() * 100:5.1f}% of rows have a real anomaly "
              f"(rest -> 0 = 'normal'), mean {df['ranomhat'].mean():+.3f} sd {df['ranomhat'].std():.3f}")
else:
    for df in (train_df, test_forecast_df, test_spatial_df, ic_df, collocation):
        df["ranomhat"] = 0.0

# --- Run 22: which columns actually reach the network -------------------------------------
# EXTRA_COLS are plain covariates.  ANCHOR_COLS are the per-well features that are dropped out
# together (see anchor_dropout) and whose FIRST column is the additive residual base -- so
# h0hat must stay first there.
# --- Step 4: Antecedent Precipitation Index (API-30, API-90) ---
# API_k(t) = sum_{tau} R(t-tau)*exp(-tau/k); captures infiltration lag in TN monsoon.
# Built from the R proxy in train observations (train only, no future leakage).
# Vectorized via 1D temporal convolution per district grid (<1s across all splits).
# Held-out wells and collocation get 0 (same convention as h0hat).
if USE_API:
    import time as _time
    t0_api = _time.time()
    _obs_r = train_df.dropna(subset=['R']).copy()
    _obs_r['t_int'] = _obs_r['t'].astype(int)
    _dist_day = _obs_r.groupby(['district', 't_int'])['R'].mean().unstack(fill_value=0.0)
    _all_days = np.arange(_dist_day.columns.min(), _dist_day.columns.max() + 1)
    _dist_day = _dist_day.reindex(columns=_all_days, fill_value=0.0)

    for k in API_DECAY_DAYS:
        hcol = f'api{k}hat'
        _max_lag = int(k * 5)
        _weights = np.exp(-np.arange(_max_lag) / k)
        _grid = np.array([np.convolve(row, _weights, mode='full')[:len(row)] for row in _dist_day.values])
        _api_df = pd.DataFrame(_grid, index=_dist_day.index, columns=_dist_day.columns)
        _lookup = _api_df.stack()

        def _map_api(df):
            _key = pd.MultiIndex.from_arrays([df['district'], df['t'].astype(int)])
            return _lookup.reindex(_key).fillna(0.0).to_numpy(dtype=np.float32)

        _train_raw = _map_api(train_df)
        ac = float(_train_raw.mean())
        asd = float(_train_raw.std()) or 1.0
        train_df[hcol] = (_train_raw - ac) / asd

        for _df in (test_forecast_df, test_spatial_df, ic_df):
            _df[hcol] = (_map_api(_df) - ac) / asd
        collocation[hcol] = 0.0
        print(f'API-{k}: c={ac:.3f} s={asd:.3f} nonzero {100*(_train_raw!=0).mean():.1f}% [{_time.time()-t0_api:.2f}s]')
else:
    for k in API_DECAY_DAYS:
        for _df in (train_df, test_forecast_df, test_spatial_df, ic_df, collocation):
            _df[f'api{k}hat'] = 0.0
    print(f'USE_API=False: API columns zeroed')

EXTRA_COLS = ["elevhat", "Rhat", "Qhat"] + (["ranomhat"] if USE_RAIN_ANOMALY else []) + ([f"api{k}hat" for k in API_DECAY_DAYS] if USE_API else [])
ANCHOR_COLS = ["h0hat"] + (["velhat"] if USE_VELOCITY else [])
N_SEASONAL = 2 if USE_SEASONAL else 0
N_INPUT_FEATURES = 3 + N_SEASONAL + len(EXTRA_COLS) + len(ANCHOR_COLS)   # x,y,t + seas + extra + anchor
THAT_TRAIN_MAX = float((t_cutoff - t_c) / t_s)
print(f"network inputs: x,y,t + {N_SEASONAL} seasonal + {EXTRA_COLS} + {ANCHOR_COLS} "
      f"= {N_INPUT_FEATURES} + embedding")
print(f"  CLAMP_T_TO_TRAIN={CLAMP_T_TO_TRAIN} (that clamped at {THAT_TRAIN_MAX:+.3f} = t_cutoff)")

colloc_df = collocation.dropna(subset=["R"]).reset_index(drop=True)
print(f"{len(colloc_df):,} of {len(collocation):,} collocation rows have a real R value "
      f"and are used for the PDE loss; the rest (districts with no humidity coverage) are dropped, not zero-filled")

# Run 12: R is derived from OBSERVED rainfall, so collocation points at t >= t_cutoff would leak
# future forcing into the weights and inflate the forecast metric. Cut them.
# This deliberately leaves the forecast window physics-unconstrained -- the network extrapolates
# there on its own. The alternative (keep future collocation and treat R as a known scenario
# input) is defensible for a "what-if given this rainfall" model, but must NOT be used for a
# headline forecast number, so it is not the default here.
n_colloc_before = len(colloc_df)
colloc_df = colloc_df[colloc_df["t"] < t_cutoff].reset_index(drop=True)
print(f"collocation restricted to t < t_cutoff for the forecast split: dropped "
      f"{n_colloc_before - len(colloc_df):,} of {n_colloc_before:,} rows; {len(colloc_df):,} remain")
colloc_df["h0hat"] = 0.0   # collocation points are not monitored wells (no anchor)
if USE_VELOCITY:
    colloc_df["velhat"] = 0.0   # ...and no velocity, for the same reason

# --- temporal importance weights for training batches ---------------------------
# 3.75M of 4.58M observation rows are 6-hourly telemetry from 2021-2026, so a
# uniform random batch would see pre-2021 manual readings very rarely.  Weights
# are inverse to the number of training rows per calendar year, so each year
# contributes roughly equal gradient signal regardless of how densely sampled it is.
train_df["year"] = (train_df["t"] / 365.25 + 1970).astype(int)  # approx year from days-since-epoch
year_counts = train_df["year"].value_counts()
inv_freq = 1.0 / year_counts
train_weights_np = train_df["year"].map(inv_freq).to_numpy().astype("float32")
train_weights_np /= train_weights_np.sum()   # normalize to sum=1 for multinomial
train_weights = torch.tensor(train_weights_np, device=device)
print(f"Temporal importance weights: {year_counts.min()} (min) .. {year_counts.max()} (max) rows/year "
      f"across {len(year_counts)} years -- sampling would equalize year-level contribution")
# Run 25: report what each sampler actually spends the gradient on, so the trade is visible in
# the log rather than inferred.  `recent` is the 2021+ telemetry era, which is where the whole
# forecast evaluation lives.
recent = (train_df["year"] >= 2021).to_numpy()
print(f"  2021+ rows: {recent.sum():,} of {len(train_df):,} ({recent.mean() * 100:.1f}% of rows) -> "
      f"{train_weights_np[recent].sum() * 100:.1f}% of gradient weighted / "
      f"{recent.mean() * 100:.1f}% uniform")
if not USE_TEMPORAL_WEIGHTS:
    train_weights = None
    print("  USE_TEMPORAL_WEIGHTS=False -> UNIFORM sampling (sample_batch falls back to randint)")
else:
    print("  USE_TEMPORAL_WEIGHTS=True -> weighted sampling")

In [ ]:
def to_tensor(df, cols, dtype=torch.float32):
    return torch.tensor(df[cols].to_numpy(), dtype=dtype, device=device)

# Run 22: covariates are stacked into two blocks instead of passed as a dozen positional args.
#   extra  -- plain covariates (EXTRA_COLS), never differentiated in pde_residual
#   anchor -- per-well features (ANCHOR_COLS), dropped out together; column 0 IS h0hat, the
#             additive base of the residual-from-anchor form, so its position is load-bearing
# x/y/t stay separate because pde_residual needs requires_grad_ on exactly those three.
# Seasonal features are NOT here -- the model derives them from `that` (see the Model cell).
def make_data_tensors(df):
    return (to_tensor(df, ["xhat"]), to_tensor(df, ["yhat"]), to_tensor(df, ["that"]),
            to_tensor(df, EXTRA_COLS), to_tensor(df, ANCHOR_COLS),
            torch.tensor(df["district_idx"].to_numpy(), dtype=torch.long, device=device),
            to_tensor(df, ["hhat"]))   # normalized depth target; the eval cell converts back to meters

def make_colloc_tensors(df):
    return (to_tensor(df, ["xhat"]), to_tensor(df, ["yhat"]), to_tensor(df, ["that"]),
            to_tensor(df, EXTRA_COLS), to_tensor(df, ANCHOR_COLS),
            torch.tensor(df["district_idx"].to_numpy(), dtype=torch.long, device=device),
            to_tensor(df, ["R"]), to_tensor(df, ["Q"]))

train_tensors = make_data_tensors(train_df)
test_forecast_tensors = make_data_tensors(test_forecast_df)
test_spatial_tensors = make_data_tensors(test_spatial_df)
ic_tensors = make_data_tensors(ic_df)
colloc_tensors = make_colloc_tensors(colloc_df)
print(f"All tensors moved to {device}; extra={EXTRA_COLS} anchor={ANCHOR_COLS}")


## Model

MLP: `(xhat, yhat, that, elevhat, rhat, qhat, district_embedding) -> hhat`, where `hhat` is
normalized depth-to-water (Run 11 switched from head to depth target). `elevhat` is fed in as a
plain input feature
(normalized like `xhat`/`yhat`/`that`), not just used to build the target -- Tamil Nadu's ~2,400m
of relief would otherwise have to be blindly reconstructed by the network from `(x, y, t,
district)` alone, which is what broke the first head-conversion run (see elevation lookup cell).
`S` and `T` are one learnable scalar per district each (the inverse-problem unknowns), passed
through `softplus` to keep them positive -- transmissivity and storage coefficient are physically
non-negative.

**Update (post Run 8)**: hidden-layer activation is now `SiLU`, not `Tanh`. Run 8 raised `W_PDE`
100x (`1e-6 -> 1e-4`) and `T` stayed byte-identical to Run 7's frozen `softplus(0)` init across
all 38 districts -- the PDE-loss curve was flat noise for the full 20,000 steps, not slowly
converging. `T` only receives gradient through the Laplacian term in `pde_residual()`; `Tanh`'s
second derivative is exactly zero at its own zero (`tanh''(0) = 0`), and `WEIGHT_DECAY` keeps
this network's pre-activations small (near that zero) almost everywhere, so the Laplacian -- and
therefore `T`'s only gradient path -- was structurally starved regardless of loss weight. `SiLU`
has the same smoothness and doesn't share that zero-curvature point, so it should let a real
Laplacian develop if one is representable at all. See the training-loop cell for the diagnostics
(`|laplacian|`, `|T_raw.grad|`) added to confirm this before trusting any `S`/`T` output.

In [ ]:
class PINN(nn.Module):
    def __init__(self, num_districts, emb_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_WIDTH, num_hidden_layers=NUM_HIDDEN_LAYERS):
        super().__init__()
        self.district_embedding = nn.Embedding(num_districts, emb_dim)
        # Run 20: RESIDUAL-FROM-ANCHOR form.  The MLP predicts the normalized DEVIATION
        # (h - h0) / h_s from the well's pre-cutoff anchor, and forward() adds the anchor
        # back:  hhat = h0hat + mlp(...).  At init (mlp ~ 0) the model IS persistence, so
        # the loss gradient goes entirely into learning the trend on top of the anchor --
        # the direct attack on the forecast gap to persistence.  h0hat stays in the input
        # too, so the net can tell whether it is anchored: with H0_DROPOUT_P it also sees
        # the unanchored (h0hat=0) case in training, where its output must be the absolute
        # field -- which is exactly what held-out wells and collocation need.
        # Run 22: the input width is now derived from the feature flags (N_INPUT_FEATURES,
        # set in the preprocessing cell) instead of hardcoded, so toggling a feature does
        # not silently mis-shape the first layer.
        layers = []
        prev = N_INPUT_FEATURES + emb_dim
        for _ in range(num_hidden_layers):
            layers.append(nn.Linear(prev, hidden_dim))
            layers.append(nn.SiLU())   # swapped from Tanh after Run 8 (STATUS.md): Tanh's second
                                        # derivative is exactly zero at z=0 (tanh''(0) = -2*tanh(0)*(1-tanh(0)^2) = 0),
                                        # and WEIGHT_DECAY keeps pre-activations near 0 almost everywhere, so the
                                        # Laplacian term in pde_residual() vanished and starved T's only gradient
                                        # path regardless of W_PDE. SiLU''(0) = 0.5 != 0 -- same smoothness/domain,
                                        # no special init required, but doesn't share Tanh's zero-curvature point.
            prev = hidden_dim
        layers.append(nn.Linear(prev, 1))
        self.mlp = nn.Sequential(*layers)
        # inverse-problem parameters: raw (pre-softplus) S, T, one scalar per district
        self.S_raw = nn.Parameter(torch.zeros(num_districts))
        self.T_raw = nn.Parameter(torch.zeros(num_districts))
        # Step 1: learnable delta scale. Init at log(0.3) -> softplus ~0.30 (Run 31 optimum).
        # Excluded from weight_decay (same group as S_raw/T_raw).
        import math as _math
        self.log_alpha = nn.Parameter(torch.tensor(_math.log(0.3))) if USE_ALPHA_SCALE else None

    def forward(self, xhat, yhat, that, extra, anchor, district_idx):
        emb = self.district_embedding(district_idx)
        h0hat = anchor[:, :1]                      # column 0 of the anchor block, by construction
        # Run 22: CLAMP_T_TO_TRAIN feeds the MLP min(that, that_at_cutoff).  Everything past
        # the cutoff then shares one value of the trend input, so the net holds the learned
        # long-term drift at its last OBSERVED level instead of extrapolating it -- the
        # seasonal/rainfall features carry the forecast window instead.  The seasonal phase
        # below deliberately uses the UNCLAMPED `that`: the annual cycle does keep running.
        that_in = torch.clamp(that, max=THAT_TRAIN_MAX) if CLAMP_T_TO_TRAIN else that
        parts = [xhat, yhat, that_in]
        if USE_SEASONAL:
            phase = SEAS_W * that + SEAS_B         # differentiable in `that` -> dh/dt is complete
            parts += [torch.sin(phase), torch.cos(phase)]
        parts += [extra, anchor, emb]
        delta = self.mlp(torch.cat(parts, dim=1))            # residual-from-anchor: mlp learns h - h0
        if USE_ALPHA_SCALE and self.log_alpha is not None:
            # Step 1: scale delta by learned alpha (init ~0.30, Run 31 post-hoc optimum)
            delta = F.softplus(self.log_alpha) * delta
        return h0hat + delta

    def get_S(self, district_idx):
        return F.softplus(self.S_raw[district_idx]).unsqueeze(1)

    def get_T(self, district_idx):
        return F.softplus(self.T_raw[district_idx]).unsqueeze(1)


model = PINN(NUM_DISTRICTS).to(device)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## PDE residual

`S(x,y) * dh/dt = div(T(x,y) * grad(h)) + R - Q`. `S` and `T` are per-district constants (not
smooth spatial fields), so their own spatial gradient is zero within a district and
`div(T * grad(h))` simplifies to `T * laplacian(h)` -- this drops any boundary-flux term at
district edges, a modeling simplification worth being aware of, not a bug.

**Update**: this equation governs *hydraulic head*, and the network now predicts head (`elev - h`,
see the elevation lookup cell), not raw depth-to-water -- previously `h` here was depth, so two
adjacent wells with different ground elevations had different depth-to-water but the same head,
and a chunk of the spatial variation the PDE "saw" was really surface topography (STATUS.md issue
2/4, the leading suspect for why `pde` never converged in any prior run). `elevhat` is passed into
the model as a **fixed covariate** here -- `xhat`/`yhat`/`that` get `requires_grad_(True)` so
`torch.autograd.grad` can differentiate head w.r.t. them, but `elevhat` does not, so `d(head)/dx`
means "how head changes moving in x while holding this point's elevation fixed," the standard way
to feed an auxiliary field into a PINN without it absorbing the PDE's own spatial derivatives.
One caveat remains: SRTM is a surface model, not bare-earth -- in canopy/urban cells it measures
treetop/rooftop height, not true ground elevation, so `elev` is an approximation.

Second derivatives are computed via `torch.autograd.grad` with `create_graph=True` (so the
graph survives for the backward pass through the loss). Inputs are normalized (`xhat`, `yhat`,
`that`) and so is the output (`headhat`), so derivatives are taken w.r.t. those and converted back
to physical units via the chain rule: `d/dx_m = (1/x_s) * d/dxhat` and `dhead = head_s * dheadhat`,
hence `d^2head/dx_m^2 = (head_s / x_s^2) * d^2headhat/dxhat^2`.

**Update (Run 9, the diffusion-term fix)**: `pde_residual` now returns `laplacian` alongside
`residual`, purely as a training-time diagnostic (see training-loop/loss-curves cells), and the
residual multiplies the diffusion term by `LAPLACIAN_SCALE` (config cell). A local smoke test with
the new diagnostic showed why Run 8 (`W_PDE` raised 100x) still left `T` byte-identical to Run 7's
frozen init: the chain-rule factor `head_s / x_s^2` (head varies ~200m; `x` spans Tamil Nadu's
~700km width in meters, so `x_s ~ 91,000m`) shrinks the physical Laplacian to ~1e-11, about 11
orders of magnitude smaller than `R`/`Q` -- `T`'s only gradient path (`-T_d * laplacian`) was
structurally invisible to the optimizer, independent of `W_PDE` or activation choice.
`LAPLACIAN_SCALE` counteracts this by scaling only the diffusion term's contribution to the
residual (not `T_d`'s own parametrization, which stays `softplus(T_raw)` as before) -- see the
config cell's full derivation and the important caveat there: `R`/`Q` are documented proxies
without true SI calibration, so the resulting `T` should be read as a *relative* quantity
(compare districts to each other) rather than literal m²/day transmissivity.

In [ ]:
def pde_residual(model, xhat, yhat, that, extra, anchor, district_idx, R, Q):
    xhat = xhat.clone().requires_grad_(True)
    yhat = yhat.clone().requires_grad_(True)
    that = that.clone().requires_grad_(True)
    # `extra` and `anchor` are NOT made to require grad -- they're fixed covariates, not
    # differentiated (see pde_residual rationale cells). Collocation points carry an all-zero
    # anchor (no well), so this is the pure physics field there.
    # Run 22: the seasonal sin/cos are computed INSIDE forward() from `that`, so the dh/dt
    # below now includes the seasonal term through the chain rule.  Passing them as constant
    # inputs instead would have left dh/dt missing that term entirely.

    hhat = model(xhat, yhat, that, extra, anchor, district_idx)   # normalized depth
    ones = torch.ones_like(hhat)

    dhhat_dxhat = torch.autograd.grad(hhat, xhat, grad_outputs=ones, create_graph=True)[0]
    dhhat_dyhat = torch.autograd.grad(hhat, yhat, grad_outputs=ones, create_graph=True)[0]
    dhhat_dthat = torch.autograd.grad(hhat, that, grad_outputs=ones, create_graph=True)[0]

    d2hhat_dxhat2 = torch.autograd.grad(dhhat_dxhat, xhat, grad_outputs=torch.ones_like(dhhat_dxhat), create_graph=True)[0]
    d2hhat_dyhat2 = torch.autograd.grad(dhhat_dyhat, yhat, grad_outputs=torch.ones_like(dhhat_dyhat), create_graph=True)[0]

    # Chain rule back to physical units, then depth -> head. The network predicts normalized
    # hhat, so h = hhat*h_s + h_c and d h/d(.) = h_s * d hhat/d(.) -- the additive h_c drops out
    # of every derivative.
    dh_dt = h_s * dhhat_dthat / t_s                          # d h(m) / d t(days)
    d2h_dx2 = h_s * d2hhat_dxhat2 / (x_s ** 2)                # d2 h(m) / d x(m)^2
    d2h_dy2 = h_s * d2hhat_dyhat2 / (y_s ** 2)                # d2 h(m) / d y(m)^2

    # head = elev - h. This residual is ALGEBRAICALLY IDENTICAL to Run 10's, not an approximation:
    #  - elev is time-invariant, so d head/dt = -d h/dt exactly.
    #  - `elevhat` is deliberately excluded from the autograd graph above (it always has been --
    #    see the comment at the top of this function), so Run 10's `laplacian` was already the
    #    fixed-elev PARTIAL second derivative of head, and at fixed elev d2 head/dx2 = -d2 h/dx2.
    #    The terrain-curvature term d2 elev/dx2 was never in the residual and is not being
    #    dropped here; switching the target does not lose any physics that was previously present.
    # Net effect: both terms flip sign relative to Run 10, and their magnitudes are unchanged.
    dhead_dt = -dh_dt
    laplacian = -(d2h_dx2 + d2h_dy2)

    S_d = model.get_S(district_idx)
    T_d = model.get_T(district_idx)

    # S*dhead/dt - T*laplacian(head) - R + Q == 0 at a true solution
    # LAPLACIAN_SCALE (config cell, added for Run 9) multiplies only the diffusion term's
    # CONTRIBUTION TO THE RESIDUAL, not laplacian itself or T_d's parametrization -- see the
    # config cell's long comment for why: the physical `laplacian` is ~1e-11 (chain-rule
    # shrinkage over Tamil Nadu's ~700km extent), ~11 orders of magnitude smaller than R/Q,
    # so T's gradient was invisible to the optimizer regardless of W_PDE or activation choice.
    residual = S_d * dhead_dt - LAPLACIAN_SCALE * T_d * laplacian - R + Q
    # laplacian is returned UNSCALED, purely as a training-time diagnostic (see training loop
    # cell) -- this keeps the printed/plotted |laplacian| interpretable as the actual physical
    # second derivative, separate from whatever multiplier the residual applies to make T
    # learnable.
    return residual, laplacian


## Loss

`L = W_DATA * MSE(h_pred, h_true) + W_PDE * MSE(pde_residual) + W_IC * MSE(ic_mismatch)`, components logged separately.
Two extra diagnostics are logged alongside (not part of the loss): `laplacian_abs_mean` and
`t_grad_norm` (`T_raw.grad.norm()`), added after Run 8 left `T` frozen with no visible signal in
the existing loss curves as to why -- see the PDE residual and training-loop cells.

In [ ]:
def sample_batch(tensors, batch_size, generator=None, weights=None):
    """Sample a mini-batch from a tuple of tensors (all same first dimension).

    If `weights` is provided (a 1-D float tensor of the same length), samples
    are drawn proportionally to those weights -- used for temporal importance
    sampling so that years with fewer readings contribute equally to each batch.
    Without weights, falls back to uniform random sampling.
    """
    n = tensors[0].shape[0]
    k = min(batch_size, n)
    if weights is not None:
        idx = torch.multinomial(weights, k, replacement=True, generator=generator)
    else:
        idx = torch.randint(0, n, (k,), device=device, generator=generator)
    return tuple(t[idx] for t in tensors)

def anchor_dropout(anchor, return_mask=False):
    """Run 20: zero the anchor on a random H0_DROPOUT_P fraction of TRAINING rows, in
    BOTH the input slot AND the additive residual (same tensor).  forward() returns
    h0hat + mlp(...), so zeroing makes the MLP predict the absolute field for that row --
    which is exactly the held-out-well and collocation case.  This keeps h0hat=0 in
    distribution so the spatial metric no longer regresses (Run 19: 13.45m).

    Run 22: drops the WHOLE anchor block (h0hat and, if enabled, velhat) on the same rows.
    Dropping the level but keeping the velocity would leave a row saying "I don't know where
    this well is, but I know it's falling 2m/yr" -- a combination no held-out well can ever
    present, so the net would learn on a distribution it is never evaluated on.

    Run 24: optionally returns the keep mask so compute_losses can report the data loss
    separately for anchored and dropped rows -- see the comment there.  The mask changes
    nothing about the loss or the gradients; it is reporting only."""
    if H0_DROPOUT_P > 0 and model.training:
        keep = (torch.rand(anchor.shape[0], 1, device=anchor.device) >= H0_DROPOUT_P).float()
        return (anchor * keep, keep) if return_mask else anchor * keep
    keep = torch.ones(anchor.shape[0], 1, device=anchor.device)
    return (anchor, keep) if return_mask else anchor

def compute_losses(model, data_batch, colloc_batch, ic_batch):
    xhat, yhat, that, extra, anchor, d_idx, h_true = data_batch
    anchor_used, keep = anchor_dropout(anchor, return_mask=True)
    h_pred = model(xhat, yhat, that, extra, anchor_used, d_idx)
    sq_err = (h_pred - h_true) ** 2
    data_loss = sq_err.mean()
    # Run 24 DIAGNOSTIC (reporting only -- data_loss above is unchanged).  The single `data`
    # number mixes two completely different regimes: ~70% of rows keep their anchor (the
    # forecast regime, whose ceiling is the ~3.7m oracle floor) and ~30% have it zeroed (the
    # held-out-well regime, whose ceiling is ~11-12m and cannot improve -- it is at its feature
    # ceiling, STATUS.md Run 12).  Every run through 23 read the mixture as if it were the
    # training fit.  Split it so we can finally see whether the anchored fit is good and the
    # model fails to generalize forward in time, or whether it never fits at all -- and how
    # much of the data gradient is being spent on the unimprovable half.
    n_keep = keep.sum()
    n_drop = keep.numel() - n_keep
    data_anch = ((sq_err * keep).sum() / n_keep.clamp(min=1.0)).item()
    data_drop = ((sq_err * (1.0 - keep)).sum() / n_drop.clamp(min=1.0)).item()
    # Run 21: delta regularization.  forward() returns h0hat + mlp(...), so the MLP's
    # deviation-from-anchor IS h_pred - h0hat_used.  Penalizing its mean square shrinks
    # the learned trend toward zero unless the data justifies it (see W_DELTA config).
    delta = h_pred - anchor_used[:, :1]
    delta_loss = torch.mean(delta ** 2)

    xhat_i, yhat_i, that_i, extra_i, anchor_i, d_idx_i, h_true_i = ic_batch
    h_pred_i = model(xhat_i, yhat_i, that_i, extra_i, anchor_dropout(anchor_i), d_idx_i)
    ic_loss = F.mse_loss(h_pred_i, h_true_i)

    xhat_c, yhat_c, that_c, extra_c, anchor_c, d_idx_c, R_c, Q_c = colloc_batch
    residual, laplacian = pde_residual(model, xhat_c, yhat_c, that_c, extra_c, anchor_c, d_idx_c, R_c, Q_c)
    pde_loss = torch.mean(residual ** 2)
    # diagnostic only -- not part of the loss (see pde_residual and training-loop cells)
    laplacian_abs_mean = laplacian.detach().abs().mean().item()

    total = W_DATA * data_loss + W_PDE * pde_loss + W_IC * ic_loss + W_DELTA * delta_loss
    components = {"data": data_loss.item(), "pde": pde_loss.item(), "ic": ic_loss.item(), "delta": delta_loss.item(), "total": total.item(),
                  "data_anch": data_anch, "data_drop": data_drop,
                  "laplacian_abs_mean": laplacian_abs_mean}
    return total, components


## Training loop

Adam + step LR schedule, loss components printed every `PRINT_EVERY` epochs.

In [ ]:
# Run 11 fix (STATUS.md tenth-run entry): S_raw/T_raw are PHYSICAL inverse-problem parameters,
# not network weights, and were previously caught by the blanket weight_decay on
# model.parameters(). Weight decay pulls *_raw toward 0, i.e. pulls S/T toward the arbitrary
# constant softplus(0)=0.6931 -- regularizing a physical quantity toward an arbitrary value is
# wrong regardless of scale. T escaped it in Run 10 only because LAPLACIAN_SCALE made its
# gradient big enough to overcome the pull; S, whose gradient is near zero, was held at init by
# it. Split into param groups so decay still regularizes the MLP (where it is doing real work
# against well memorization -- see WEIGHT_DECAY) but not S_raw/T_raw.
physical_params = [model.S_raw, model.T_raw] + ([model.log_alpha] if USE_ALPHA_SCALE and hasattr(model, "log_alpha") and model.log_alpha is not None else [])  # Step 1: log_alpha is a physical scalar, not a network weight
physical_ids = {id(p) for p in physical_params}
network_params = [p for p in model.parameters() if id(p) not in physical_ids]
optimizer = torch.optim.Adam(
    [{"params": network_params, "weight_decay": WEIGHT_DECAY},
     {"params": physical_params, "weight_decay": 0.0}],
    lr=ADAM_LR,
)
print(f"optimizer: {sum(p.numel() for p in network_params):,} network params "
      f"(weight_decay={WEIGHT_DECAY}) + {sum(p.numel() for p in physical_params):,} physical "
      f"S/T params + {1 if USE_ALPHA_SCALE else 0} alpha param (weight_decay=0)")
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)

history = {"data": [], "pde": [], "ic": [], "delta": [], "total": [], "laplacian_abs_mean": [],
           "data_anch": [], "data_drop": [], "t_grad_norm": [], "s_grad_norm": []}

model.train()
for epoch in range(N_EPOCHS_ADAM):
    data_batch = sample_batch(train_tensors, BATCH_SIZE_DATA, weights=train_weights)
    colloc_batch = sample_batch(colloc_tensors, BATCH_SIZE_COLLOC)
    ic_batch = sample_batch(ic_tensors, BATCH_SIZE_IC)

    optimizer.zero_grad()
    total_loss, components = compute_losses(model, data_batch, colloc_batch, ic_batch)
    total_loss.backward()

    # Diagnostic for the Run 8 finding (STATUS.md): T only gets gradient through the Laplacian
    # term in the PDE residual, and Run 8 (W_PDE raised 100x, 1e-6 -> 1e-4) left T byte-identical
    # to Run 7's frozen init anyway -- consistent with a near-zero Laplacian starving T's gradient
    # regardless of loss weight. Logged every step (cheap, one .norm() call) so loss_curves.png
    # shows directly whether the Tanh -> SiLU swap above fixed it, instead of inferring it after
    # the fact from a suspiciously round S/T value the way Run 8 was diagnosed.
    t_grad_norm = model.T_raw.grad.norm().item() if model.T_raw.grad is not None else float("nan")
    components["t_grad_norm"] = t_grad_norm
    # Run 11: the S-side twin of the above. Run 10 left S within 4.3e-04 of softplus(0) while T
    # moved 3.135 (7,300x further), and the leading hypothesis is that S's only gradient path --
    # the UNSCALED `S_d * dhead_dt` term in pde_residual -- is starved against R/Q at O(1-20),
    # the same way T's was before LAPLACIAN_SCALE. That was never measured, only inferred. This
    # settles it: if |S_raw.grad| is ~1e-14 float noise, S needs its own scale multiplier
    # (Next step 2); if it is healthy and S still doesn't move, the diagnosis is wrong and the
    # weight-decay fix above was the whole story.
    s_grad_norm = model.S_raw.grad.norm().item() if model.S_raw.grad is not None else float("nan")
    components["s_grad_norm"] = s_grad_norm

    optimizer.step()
    scheduler.step()

    for k, v in components.items():
        history.setdefault(k, []).append(v)

    if epoch % PRINT_EVERY == 0 or epoch == N_EPOCHS_ADAM - 1:
        lr_now = scheduler.get_last_lr()[0]
        # data/ic are MSE in NORMALIZED DEPTH units (Run 11); the meters-of-depth equivalent is
        # printed alongside. This IS now directly comparable to the held-out RMSE in the eval cell
        # (same quantity, no elev round-trip) -- the gap between them is the generalization gap.
        data_rmse_h_m = (components["data"] ** 0.5) * h_s
        # Run 24: anch/drop are the same MSE restricted to anchored / anchor-dropped rows.
        anch_m = (components["data_anch"] ** 0.5) * h_s
        drop_m = (components["data_drop"] ** 0.5) * h_s
        print(f"epoch {epoch:6d}  lr={lr_now:.2e}  data={components['data']:.4f} ({data_rmse_h_m:5.2f}m depth"
              f" = anch {anch_m:5.2f}m / drop {drop_m:5.2f}m)  "
              f"pde={components['pde']:.4f}  ic={components['ic']:.4f}  delta={components['delta']:.4f}  total={components['total']:.4f}  "
              f"|laplacian|={components['laplacian_abs_mean']:.2e}  |T_raw.grad|={t_grad_norm:.2e}  "
              f"|S_raw.grad|={s_grad_norm:.2e}")

## Optional L-BFGS fine-tune

A short second-order pass on a large fixed batch, common for PINNs once Adam has roughly converged.

In [ ]:
if RUN_LBFGS_FINETUNE:
    lbfgs_data = sample_batch(train_tensors, min(20000, train_tensors[0].shape[0]))
    lbfgs_colloc = sample_batch(colloc_tensors, min(20000, colloc_tensors[0].shape[0]))
    lbfgs_ic = sample_batch(ic_tensors, min(4000, ic_tensors[0].shape[0]))

    lbfgs_opt = torch.optim.LBFGS(model.parameters(), max_iter=LBFGS_MAX_ITER,
                                  line_search_fn="strong_wolfe")

    def closure():
        lbfgs_opt.zero_grad()
        loss, components = compute_losses(model, lbfgs_data, lbfgs_colloc, lbfgs_ic)
        loss.backward()
        # same diagnostic as the Adam loop -- must come after backward() for T_raw.grad to exist
        components["t_grad_norm"] = model.T_raw.grad.norm().item() if model.T_raw.grad is not None else float("nan")
        components["s_grad_norm"] = model.S_raw.grad.norm().item() if model.S_raw.grad is not None else float("nan")
        closure.last_components = components
        return loss

    lbfgs_opt.step(closure)
    c = closure.last_components
    print(f"L-BFGS finished: data={c['data']:.4f}  pde={c['pde']:.4f}  ic={c['ic']:.4f}  total={c['total']:.4f}  "
          f"|laplacian|={c['laplacian_abs_mean']:.2e}  |T_raw.grad|={c['t_grad_norm']:.2e}  "
          f"|S_raw.grad|={c['s_grad_norm']:.2e}")
    for k in history:
        history[k].append(c[k])
else:
    print("Skipped L-BFGS fine-tune (RUN_LBFGS_FINETUNE=False)")


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(24, 12))
# Run 24: data_anch / data_drop are `data` restricted to anchored / anchor-dropped rows.
# Plotted next to `data` because `data` is their sample-frequency average (0.7/0.3) and had
# been read as "the training fit" for four runs -- see STATUS.md Run 22/24.
loss_keys = ["data", "data_anch", "data_drop", "pde", "ic", "delta", "total"]
diag_keys = ["laplacian_abs_mean", "t_grad_norm", "s_grad_norm"]   # Run 8/11 diagnostics, see pde_residual/training-loop cells
for ax, key in zip(axes.ravel(), loss_keys + diag_keys):
    ax.plot(history[key])
    if key in loss_keys:
        ax.set_yscale("log")
        ax.set_title(f"{key} loss")
    else:
        ax.set_title(key)   # linear scale -- these can legitimately hit 0, which log-scale can't show
    ax.set_xlabel("step")
for ax in axes.ravel()[len(loss_keys) + len(diag_keys):]:
    ax.axis("off")   # 3x4 grid, 10 series -- hide the spare slots
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "loss_curves.png"), dpi=120)
plt.show()


## Evaluation on held-out wells

Held out **by well** (see preprocessing cell), not by row, so no station leaks between train
and test across time. The network predicts normalized head; predictions are converted back to
head in meters (`* head_s + head_c`), then to **depth-to-water** via `elev` (`h_pred = elev -
head_pred`) so this run's RMSE/MAE stay in the same units as every prior run and the baselines
below -- `h_true` is taken directly from `test_df["h"]`, not reconstructed from `elev`, so
ground-truth values carry no elevation-lookup error.

Scored against trivial baselines fitted on the training wells only. This matters: the first
three runs (RMSE 15.6-18.3m) all *lost* to predicting a single constant (13.3m) or the district
mean (12.95m), which is not visible from the loss curves alone. A PINN that can't beat the
district mean isn't learning a useful field, whatever its loss says.

In [ ]:
model.eval()


def rmse(pred, true):
    return float(np.sqrt(np.mean((np.asarray(pred) - np.asarray(true)) ** 2)))


def mae(pred, true):
    return float(np.mean(np.abs(np.asarray(pred) - np.asarray(true))))


def predict_h(tensors):
    """Network output -> depth-to-water in meters. Direct since Run 11: the network predicts h
    itself, so nothing cancels through a ~200m elevation term on the way to the metric."""
    with torch.no_grad():
        xhat, yhat, that, extra, anchor, d_idx, _ = tensors
        hhat_pred = model(xhat, yhat, that, extra, anchor, d_idx)
    return hhat_pred.cpu().numpy().ravel() * h_s + h_c


def report(name, test_df_, tensors, baselines, note=""):
    """Score the PINN against a dict of {label: prediction array} on one test set."""
    h_pred = predict_h(tensors)
    h_true = test_df_["h"].to_numpy()
    m_rmse, m_mae = rmse(h_pred, h_true), mae(h_pred, h_true)

    print(f"\n=== {name} ===")
    print(f"{len(test_df_):,} rows over {test_df_['well_id'].nunique():,} wells" + (f"  {note}" if note else ""))
    print()
    w = max(len(l) for l in baselines) + 10
    print(f"  {'predictor':<{w}} {'RMSE (m)':>10} {'MAE (m)':>10}")
    print(f"  {'-' * (w + 22)}")
    print(f"  {'PINN':<{w}} {m_rmse:>10.3f} {m_mae:>10.3f}")
    for label, pred in baselines.items():
        print(f"  {('baseline: ' + label):<{w}} {rmse(pred, h_true):>10.3f} {mae(pred, h_true):>10.3f}")

    best_label = min(baselines, key=lambda k: rmse(baselines[k], h_true))
    best = rmse(baselines[best_label], h_true)
    print()
    if m_rmse < best:
        print(f"  PINN beats the best baseline ({best_label}) by {best - m_rmse:.3f}m RMSE.")
    else:
        print(f"  *** PINN LOSES to the best baseline ({best_label}) by {m_rmse - best:.3f}m RMSE. ***")
    return h_pred, h_true, m_rmse, m_mae, best_label, best


global_mean = train_df["h"].mean()
district_mean = train_df.groupby("district")["h"].mean()

# ---------------------------------------------------------------------------------------
# PRIMARY: forecast task -- wells the model trained on, at times it never saw.
# Baselines are all fitted on train_df only. Persistence and the final-year mean are the ones
# that matter: a groundwater forecast that cannot beat "assume it stays where it was" is not
# forecasting anything. Local diagnostics put these at ~5.61m / ~5.55m and the oracle floor
# (each well's own mean within the test period) at 3.73m.
# ---------------------------------------------------------------------------------------
fc = test_forecast_df
well_mean_tr = train_df.groupby("well_id")["h"].mean()
persistence = train_df.sort_values("t").groupby("well_id")["h"].last()
final_year = train_df[train_df["t"] >= t_cutoff - 365].groupby("well_id")["h"].mean()

fc_baselines = {
    "persistence (last obs before cutoff)": fc["well_id"].map(persistence).fillna(global_mean).to_numpy(),
    "final-year mean before cutoff": fc["well_id"].map(final_year).fillna(global_mean).to_numpy(),
    "per-well mean (train period)": fc["well_id"].map(well_mean_tr).fillna(global_mean).to_numpy(),
    "district mean": fc["district"].map(district_mean).fillna(global_mean).to_numpy(),
    "global mean": np.full(len(fc), global_mean),
}
fc_pred, fc_true, fc_rmse, fc_mae, fc_best_label, fc_best = report(
    "FORECAST (primary): known wells, t >= cutoff", fc, test_forecast_tensors, fc_baselines,
    note=f"| oracle floor {rmse(fc['well_id'].map(fc.groupby('well_id')['h'].mean()), fc['h']):.2f}m")

# Run 22 diagnostic: the headline RMSE hides WHERE the delta helps.  Run 20's local analysis
# found PINN beat persistence on only 160 of 699 wells -- the learned deviation was net-harmful.
# Printed every run now, so a feature that helps on average but poisons most wells is visible.
pers_pred = fc_baselines["persistence (last obs before cutoff)"]
fc_pw = pd.DataFrame({"well_id": fc["well_id"].to_numpy(),
                      "se_pinn": (fc_pred - fc_true) ** 2,
                      "se_pers": (pers_pred - fc_true) ** 2})
pw = fc_pw.groupby("well_id")[["se_pinn", "se_pers"]].mean() ** 0.5
n_win = int((pw["se_pinn"] < pw["se_pers"]).sum())
keep_wells = pw.index[pw["se_pinn"] <= pw["se_pinn"].quantile(0.99)]
trim = fc_pw["well_id"].isin(keep_wells).to_numpy()
print()
print(f"  per-well: PINN beats persistence on {n_win}/{len(pw)} wells "
      f"({100 * n_win / len(pw):.0f}%)")
print(f"  median per-well RMSE: PINN {pw['se_pinn'].median():.2f}m vs "
      f"persistence {pw['se_pers'].median():.2f}m")
print(f"  dropping the worst 1% of wells: PINN {rmse(fc_pred[trim], fc_true[trim]):.2f}m vs "
      f"persistence {rmse(pers_pred[trim], fc_true[trim]):.2f}m")
# --- Step 1: Delta-shrinkage sweep ---
print()
print("  --- Delta-shrinkage sweep: h_alpha = h0 + alpha * (h_pred - h0) ---")
h0_fc = pers_pred
delta_fc = fc_pred - h0_fc
print(f"  {'alpha':<8} {'RMSE (m)':>10} {'MAE (m)':>10} {'well wins':>12}")
print(f"  {'-' * 44}")
for alpha in np.linspace(0.0, 1.0, 11):
    h_alpha = h0_fc + alpha * delta_fc
    r_a = rmse(h_alpha, fc_true)
    m_a = mae(h_alpha, fc_true)
    se_a = (h_alpha - fc_true) ** 2
    pw_a = pd.DataFrame({"se_a": se_a, "se_pers": fc_pw["se_pers"]}).groupby(fc_pw["well_id"]).mean() ** 0.5
    wins_a = int((pw_a["se_a"] < pw_a["se_pers"]).sum()) if alpha > 0 else 0
    print(f"  {alpha:<8.1f} {r_a:>10.3f} {m_a:>10.3f} {wins_a:>12d}/{len(pw)}")

# --- Step 2: Characterise the forecast tail ---
pw_ext = pw.copy()
pw_ext["diff_rmse"] = pw_ext["se_pinn"] - pw_ext["se_pers"]
pw_ext["n_train"] = pw_ext.index.map(train_df.groupby("well_id").size())
pw_ext["h_std_train"] = pw_ext.index.map(train_df.groupby("well_id")["h"].std().fillna(0))
pw_ext["h0_abs"] = pw_ext.index.map(train_df.groupby("well_id")["h"].last().abs())
pw_ext["district"] = pw_ext.index.map(train_df.groupby("well_id")["district"].first())

tr_coords = train_df.groupby("well_id").agg(x=("x", "first"), y=("y", "first"))
A_tr = np.c_[tr_coords["x"] * M_PER_DEG_LON, tr_coords["y"] * M_PER_DEG_LAT]
D_tr = np.sqrt(((A_tr[:, None, :] - A_tr[None, :, :]) ** 2).sum(-1))
np.fill_diagonal(D_tr, np.inf)
dist_nearest_m = pd.Series(D_tr.min(axis=1), index=tr_coords.index)
pw_ext["dist_nearest_train_km"] = pw_ext.index.map(dist_nearest_m) / 1000.0

print()
print("  --- Forecast tail characterisation: 20 worst wells (by PINN minus persistence RMSE) ---")
worst20 = pw_ext.sort_values("diff_rmse", ascending=False).head(20)
print(worst20[["se_pinn", "se_pers", "diff_rmse", "n_train", "h_std_train", "h0_abs", "dist_nearest_train_km", "district"]].to_string())

print()
print("  --- Correlation of PINN minus persistence error (diff_rmse) with covariates ---")
for cov in ["n_train", "h_std_train", "h0_abs", "dist_nearest_train_km"]:
    corr_val = pw_ext["diff_rmse"].corr(pw_ext[cov])
    print(f"  corr(diff_rmse, {cov:<22}): {corr_val:+.3f}")


# SECONDARY: spatial task -- held-out wells, all t. Same split as Runs 7-11 (SPLIT_SEED=0).
# The kNN baseline is the one that matters here (~11.57m). With uniform batch sampling
# (USE_TEMPORAL_WEIGHTS=False, adopted Run 25), the PINN consistently beats kNN across seeds
# (mean 10.74m).
# ---------------------------------------------------------------------------------------
sp = test_spatial_df
tr_wells = train_df.groupby("well_id").agg(x=("x", "first"), y=("y", "first"), hbar=("h", "mean"))
sp_wells = sp.groupby("well_id").agg(x=("x", "first"), y=("y", "first"))
A = np.c_[tr_wells["x"] * M_PER_DEG_LON, tr_wells["y"] * M_PER_DEG_LAT]
B = np.c_[sp_wells["x"] * M_PER_DEG_LON, sp_wells["y"] * M_PER_DEG_LAT]
D = np.sqrt(((B[:, None, :] - A[None, :, :]) ** 2).sum(-1))     # meters, brute force (~470 x ~2650)
KNN_K = 5
knn_idx = np.argsort(D, axis=1)[:, :KNN_K]
knn_d = np.take_along_axis(D, knn_idx, 1)
knn_w = 1.0 / np.maximum(knn_d, 100.0)                          # inverse distance, floored at 100m
knn_hat = (tr_wells["hbar"].to_numpy()[knn_idx] * knn_w).sum(1) / knn_w.sum(1)
knn_series = pd.Series(knn_hat, index=sp_wells.index)

sp_baselines = {
    f"kNN(x,y) on train well means, k={KNN_K}": sp["well_id"].map(knn_series).fillna(global_mean).to_numpy(),
    "district mean": sp["district"].map(district_mean).fillna(global_mean).to_numpy(),
    "global mean": np.full(len(sp), global_mean),
}
sp_pred, sp_true, sp_rmse, sp_mae, sp_best_label, sp_best = report(
    "SPATIAL (secondary): held-out wells, all t", sp, test_spatial_tensors, sp_baselines,
    note="| oracle floor 5.07m | uniform sampling beats kNN, see STATUS.md")

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax, (title, ht, hp, r, m, bl, bv) in zip(axes, [
        ("FORECAST (primary)", fc_true, fc_pred, fc_rmse, fc_mae, fc_best_label, fc_best),
        ("SPATIAL (secondary)", sp_true, sp_pred, sp_rmse, sp_mae, sp_best_label, sp_best)]):
    ax.scatter(ht, hp, s=2, alpha=0.3)
    lims = [min(ht.min(), hp.min()), max(ht.max(), hp.max())]
    ax.plot(lims, lims, "r--", linewidth=1)
    ax.set_xlabel("True h (m, depth to water)")
    ax.set_ylabel("Predicted h (m)")
    ax.set_title(f"{title}: RMSE={r:.2f}m, MAE={m:.2f}m\nbest baseline ({bl}): {bv:.2f}m", fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "pred_vs_actual.png"), dpi=120)
plt.show()


## Spatial maps at fixed time slices

Uses the collocation grid's spatial layout at 3 chosen timestamps, across all 38 districts.
R and Q are now network inputs (Run 13), so each grid point carries one representative R/Q
from the collocation set. Grid points with no elevation match were already dropped upstream
(elevation lookup cell).

In [ ]:
t_slices = np.quantile(observations["t"].to_numpy(), [0.1, 0.5, 0.9])
# R/Q are network inputs, not just residual terms -- take one representative per grid point.
GRID_KEYS = ["x", "y", "x_m", "y_m", "district_idx", "elev", "elevhat"]
# API hat cols are 0.0 in collocation (no well anchor there); include so grid_feats matches EXTRA_COLS
_api_cols = [f"api{k}hat" for k in API_DECAY_DAYS] if USE_API else []
grid_xy = collocation.groupby(GRID_KEYS, as_index=False)[["R", "Q", "ranomhat"] + _api_cols].first()
print(f"spatial map grid: {len(grid_xy):,} points")

def grid_col(series):
    return torch.tensor(series.to_numpy(), dtype=torch.float32, device=device).unsqueeze(1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)
for ax, t_val in zip(axes, t_slices):
    that_val = (t_val - t_c) / t_s
    xhat_g = grid_col((grid_xy["x_m"] - x_c) / x_s)
    yhat_g = grid_col((grid_xy["y_m"] - y_c) / y_s)
    that_g = torch.full_like(xhat_g, that_val)
    # Run 22: same EXTRA_COLS / ANCHOR_COLS the training tensors use, built in the same order --
    # a hand-ordered cat() here would silently feed the map a permuted feature vector.
    grid_feats = {"elevhat": grid_xy["elevhat"], "Rhat": (grid_xy["R"].fillna(R_c) - R_c) / R_s,
                  "Qhat": (grid_xy["Q"] - Q_c) / Q_s, "ranomhat": grid_xy["ranomhat"]}
    # Step 4: API hat cols are pre-normalized 0.0 in collocation; just pass through
    for _k in (API_DECAY_DAYS if USE_API else []):
        grid_feats[f"api{_k}hat"] = grid_xy[f"api{_k}hat"]
    extra_g = torch.cat([grid_col(grid_feats[c]) for c in EXTRA_COLS], dim=1)
    anchor_g = torch.zeros(len(grid_xy), len(ANCHOR_COLS), dtype=torch.float32, device=device)
    d_idx_g = torch.tensor(grid_xy["district_idx"].to_numpy(), dtype=torch.long, device=device)

    with torch.no_grad():
        hhat_g = model(xhat_g, yhat_g, that_g, extra_g, anchor_g, d_idx_g).cpu().numpy().ravel()
    h_g = hhat_g * h_s + h_c                                  # -> depth to water, meters (direct)

    sc = ax.scatter(grid_xy["x"], grid_xy["y"], c=h_g, cmap="viridis_r", s=25)
    ax.set_title(f"t={t_val:.0f}d")
    ax.set_xlabel("Longitude")
fig.colorbar(sc, ax=axes, label="Predicted h (m, depth to water)", shrink=0.8)
axes[0].set_ylabel("Latitude")
fig.savefig(os.path.join(OUTPUT_DIR, "spatial_maps.png"), dpi=120, bbox_inches="tight")
plt.show()

## Save outputs

Model weights, learned S/T per district, and the district index map (for reproducibility).

**Run 9 caveat**: `T` (not `S`) is now scaled inside the PDE residual by `LAPLACIAN_SCALE`
(config cell) so it can receive gradient at all -- see the PDE residual cell. `R`/`Q` are
proxy quantities without true SI calibration, so there was never a path to literal m²/day
transmissivity from this data; read the `T` column below as a **relative** measure (spread
across districts) rather than an absolute physical value.

In [ ]:
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "pinn_model.pth"))

with torch.no_grad():
    all_idx = torch.arange(NUM_DISTRICTS, device=device)
    S_values = F.softplus(model.S_raw).cpu().numpy()
    T_values = F.softplus(model.T_raw).cpu().numpy()

st_df = pd.DataFrame({"district": all_districts, "S": S_values, "T": T_values})
st_df.to_csv(os.path.join(OUTPUT_DIR, "learned_S_T_per_district.csv"), index=False)

pd.DataFrame({"district": all_districts, "district_idx": range(NUM_DISTRICTS)}) \
    .to_csv(os.path.join(OUTPUT_DIR, "district_index_map.csv"), index=False)

print(f"Saved model weights, learned S/T ({len(st_df)} districts), and the district index map to {OUTPUT_DIR}/")
print(st_df.sort_values('T', ascending=False).head())

# Step 1: report learned delta scale
if USE_ALPHA_SCALE and hasattr(model, 'log_alpha') and model.log_alpha is not None:
    _a = float(F.softplus(model.log_alpha).item())
    print(f'Learned delta scale alpha = {_a:.4f} (init ~0.30, Run 31 post-hoc optimum)')